# HLS × Landsat LST × Sentinel-1 × PlanetScope cover-crop workflow — IN, three seasons (v11)

**Question.** On two fields with a known cover-crop history, what do the
remote-sensing records look like in the cover-crop windows, field against
control, season by season? Three seasons of the IN site (Peng 2025):

| season | cover crop | detection window(s) |
|---|---|---|
| 2021-22 | rye drilled after soybean, 2021-11-08 → terminated 2022-05-16 | spring: 2022-03-01 → termination |
| 2022-23 | interseeded into V-stage corn 2022-06-27 → terminated 2023-05-18 | **corn**: interseeding → last corn scene; **post-harvest**: first scene after harvest → termination (harvest found in the control field's NDVI, Cell 11b) |
| 2023-24 | rye drilled after soybean, 2023-11-13 → terminated 2024-05-15 | spring: 2024-03-01 → termination |

**Descriptive only.** Two fields × three seasons cannot support a statistical
conclusion, so v11 reports values, CC − control differences and trajectories
of real observations; the v9–v10 random-forest check and sign tests are gone.

**What v11 changes from v10**

| | v10 | v11 |
|---|---|---|
| seasons | all six (2018-19 … 2023-24) | 2021-22, 2022-23, 2023-24 (`CFG["seasons"]`) |
| windows | one per season; interseeded = seeding → termination | interseeded season split at the corn harvest into a corn and a post-harvest window (Cell 11b); the v8 layers use the post-harvest window |
| radar | — | **Sentinel-1 C-band** VV, VH (RTC gamma0, Planetary Computer) on the 30 m grid (Cell 18b) |
| PlanetScope | — | CSDA inventory every run; 8-band SR download, HLS cross-calibration and field means once `CFG["ps"]["enabled"]` (Cell 18c) |
| season features | Ma et al. 144 (+ NDTI/STI, LST, raw max, Barnes fall) | same + Sentinel-1 monthly VV, VH, VH − VV (Oct–May) |
| evaluation | consistency, sign test, RF per group | per-window field summaries, CC − control (window means and same-day differences), observed trajectories (Cell 21); v8 layers per season without AUC (Cell 17) |

| cells | step |
|---|---|
| 0–4 | environment, config, labels → ROI, windows, grids |
| 5–7 | HLS six bands (`hls_bands.nc`, `hls_scenes.nc`) |
| 8–10 | ECOSTRESS ET, fused 8-day cube |
| 11, **11b** | ground-truth fields; **detection windows incl. the corn harvest** (`detect_windows.csv`) |
| 12–17 | CSB fields, cropland; v8 QC, trajectories, layers, diagnostics, layer values per season |
| 18, **18b**, **18c** | Landsat LST; **Sentinel-1** (`s1_scenes.nc`); **PlanetScope** (`ps_index.csv`, `ps_unit_obs.csv`) |
| 19–20 | Daymet; season feature table (Ma et al. 144 + NDTI/STI + LST + S1) |
| **21** | **observed trajectories and window summaries** (`window_summary.csv`, `window_paired.csv`) |
| 22–23 | archive; one-page summary |


In [ ]:
# =============================================================================
# Cell 0 — Install what a fresh container is missing (idempotent)
# =============================================================================
import importlib, subprocess, sys, site

# import name → pip name (they differ for a few packages)
PKGS = {"geopandas": "geopandas", "rioxarray": "rioxarray", "earthaccess": "earthaccess",
        "scipy": "scipy", "h5py": "h5py", "netCDF4": "netCDF4", "pyogrio": "pyogrio",
        "sklearn": "scikit-learn", "pystac_client": "pystac-client",
        "planetary_computer": "planetary-computer"}
NEEDED = list(PKGS)
# --user installs land in ~/.local/…/site-packages; on a fresh container that
# directory did not exist when this kernel started, so it is not on sys.path
# yet — add it, or every import fails until the kernel is restarted.
site.addsitedir(site.getusersitepackages())
missing = [m for m in NEEDED if importlib.util.find_spec(m) is None]
if missing:
    print("installing:", ", ".join(missing))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--user",
                        *[PKGS[m] for m in missing]], capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-2000:]
    site.addsitedir(site.getusersitepackages())
    importlib.invalidate_caches()
still = [m for m in NEEDED if importlib.util.find_spec(m) is None]
assert not still, f"still missing after install: {still}"
print("environment OK:", ", ".join(NEEDED))

## Cell 1 — Configuration

`SITES` lists the ground-truth sites to process, in order. Every later cell
loops over that list: each site gets its own ROI (its fields + 1 km — so the
HLS download is only the tiles around those fields), its own period, its own
window length and its own working/archive folders. Window length is decided by
data availability, window by window: 16 days while Landsat 8 alone feeds HLS,
8 days from the day S30 comes online (late 2015).

Also here: the CSB source, the per-season calendar of the firerx layers (`cal`) and the `green` threshold.

In [ ]:
# =============================================================================
# Cell 1 — Imports, configuration, helpers
# =============================================================================
import os, re, json, time, gc, glob, shutil, zipfile, requests
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rxr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from shapely.geometry import box, Polygon
from rasterio.enums import Resampling
from rasterio import features
import rasterio, rasterio.windows
import earthaccess
import warnings; warnings.filterwarnings("ignore")

# Validation sites with trustworthy field boundaries and paper-reported labels.
# Add a site by appending its site_id once it is in cc_groundtruth.gpkg.
SITES = ["IN_PENG2025"]

CFG = dict(
    # --- ground truth ----------------------------------------------------
    gt_gpkg   = "/data-store/iplant/home/pengyuwh/CoverCrop_Fusion/ground_truth/"
                "cc_groundtruth.gpkg",
    roi_buffer_m = 1000,        # each site's fields' bounding box grown by this much

    # --- time ------------------------------------------------------------
    # The period is derived from the labels: first detect_start minus lead,
    # last detect_end plus tail (the tail keeps the cash-crop emergence that
    # term_drop needs as its "after" state).
    lead_days = 45,
    tail_days = 75,
    x_days_4sat = 8,            # window once L30 + S30 are both flying
    x_days_1sat = 16,           # window while only Landsat 8 feeds HLS
    s30_start   = "2015-12-01", # HLS S30 v2.0 begins here (approx.)
    eco_start   = "2018-07-09", # first ECOSTRESS acquisitions

    # --- resolution / products ------------------------------------------
    res_optical = 30,
    res_thermal = 70,
    eco_product = "ECO_L3T_JET",
    eco_var     = "ET",
    cloud_max   = 80.0,
    utc_offset  = -6,           # overwritten per site from the longitude

    # --- seasons ---------------------------------------------------------
    # None = every usable season of the site. NE_BLANCO2017 carries eight
    # extra single-field seasons (2017-25, all "no CC") that would stretch the
    # run to twelve years for a few negatives; the trial years are enough.
    seasons = {"IN_PENG2025": ["2021-22", "2022-23", "2023-24"]},

    # --- field boundaries (USDA Crop Sequence Boundaries 2018-2025) ------
    csb_url    = "https://www.nass.usda.gov/Research_and_Science/Crop-Sequence-Boundaries/"
                 "datasets/NationalCSB_2018-2025_rev23.zip",
    csb_zip    = "/data-store/iplant/home/pengyuwh/Crop Sequence Boundaries/"   # fallback
                 "NationalCSB_2018-2025_rev23.zip",
    csb_dir    = "/home/jovyan/csb",      # extracted once per container, local disk
    csb_layer  = "national1825",
    crop_codes = [1, 5],                  # CDL: 1 corn, 5 soybeans
    min_field_px = 20,                    # >= 1.8 ha to count as a field for averaging

    # --- detection window per season, from the paper's cover-crop dates ------
    # One window per season, taken from that season's cover-crop record and
    # applied to every field of the site (the control is read over the same
    # dates). y0 = fall year, y1 = spring year of the season ("2021-22").
    #   fall-seeded CC (drilled after harvest): the spring after seeding,
    #       fall_start → termination        e.g. rye seeded 2021-11 → 2022-03-01 … 2022-05-16
    #   interseeded CC: seeding → termination e.g. 2022-06-27 … 2023-05-18
    #   no termination date in the paper: default_term
    windows = dict(fall_start   = "y1-03-01",
                   default_term = "y1-05-15",
                   rise_days    = 75,       # up_slope: last 75 days before termination
                   term_before  = 24,       # term_drop: termination − 24 d …
                   term_after   = 40,       #            … + 40 d (cash-crop emergence)
                   late_days    = 42,       # spring_rel, et_spring: last 6 weeks before termination
                   # interseeded seasons (Cell 11b): the corn harvest splits the season in two;
                   # found as the control field's first scene below harvest_ndvi after the peak
                   harvest_ndvi     = 0.40,
                   harvest_search   = ("y0-08-01", "y0-12-15"),
                   harvest_override = {}),  # e.g. {"2022-23": ("2022-10-14", "2022-10-29")}
    green = 0.30,                         # NDVI above bare soil + residue (provisional)

    # --- Landsat Collection 2 surface temperature (Cell 18) ----------------
    st = dict(stac        = "https://planetarycomputer.microsoft.com/api/stac/v1",
              collection  = "landsat-c2-l2",
              platforms   = ["landsat-7", "landsat-8", "landsat-9"],
              cloud_max   = 80,           # scene-level pre-filter; QA_PIXEL does the masking
              scale       = 0.00341802,   # ST DN → kelvin: DN * scale + offset
              offset      = 149.0,
              min_crop_clear = 0.20),     # a scene needs this share of cropland clear to be used

    # --- Sentinel-1 C-band backscatter (Cell 18b) --------------------------
    s1 = dict(stac       = "https://planetarycomputer.microsoft.com/api/stac/v1",
              collection = "sentinel-1-rtc"),   # IW GRD, gamma0 RTC, VV + VH, 10 m

    # --- PlanetScope through NASA CSDA (Cell 18c) --------------------------
    ps = dict(enabled    = False,     # True once the CSDA authorization is approved
              stac       = "https://csdap.earthdata.nasa.gov/stac/",
              api        = "https://csdap.earthdata.nasa.gov",
              collection = "planet",
              sr_asset   = "ortho_analytic_8b_sr",
              udm_asset  = "ortho_udm2",
              min_clear_percent = 50,  # scene-level pre-filter; UDM2 does the masking
              res        = 3,
              edge_m     = 6,          # field boundaries shrunk by this before averaging
              match_days = 1,          # HLS scene within ± days for cross-calibration
              min_cal_px = 50,
              min_cal_r  = 0.8),

    # --- Daymet weather for the Ma et al. baseline (Cell 19) ----------------
    wx = dict(api       = "https://daymet.ornl.gov/single-pixel/api/data",
              vars      = "tmax,tmin,prcp",
              round_deg = 0.0001),        # query at the centroid (~10 m); pixels are matched by their data

    # --- season feature table (Cells 20-21) -----------------------------
    # Ma et al. 2026 (IJAEOG 149:105258) baseline: Oct–May, 6 Landsat bands +
    # NDVI + GCVI, 2nd-order harmonic fit, monthly min/max. Barnes et al. 2021
    # (Remote Sens. 13:1998): fall window Oct 1 – Dec 1 with NDTI/STI + LST.
    feat = dict(window     = ("y0-10-01", "y1-05-31"),
                fall       = ("y0-10-01", "y0-12-01"),
                spring_max = ("y1-04-01", "y1-05-31"),
                months     = [10, 11, 12, 1, 2, 3, 4, 5],
                harm_order = 2,
                min_obs    = 7,           # clear scenes needed to fit the harmonic
                min_clear  = 0.5),        # share of a field's pixels that must be clear

    # --- storage ---------------------------------------------------------
    work_root    = "/home/jovyan/fusion_sites",                      # local disk
    archive_root = "/data-store/iplant/home/pengyuwh/CoverCrop_Fusion/ground_truth_runs",
    use_archive  = True,
)

# Per-site globals. Every cell below starts its loop with use_site(SITE), which
# points WORK / ARCHIVE / caches / figures at that site and loads the site's own
# period and window settings (cfg_site.json, written by Cell 3) into CFG.
SITE = WORK = ARCHIVE = CACHE_HLS = CACHE_ECO = FIG_DIR = None
P = lambda name: os.path.join(WORK, name)


# Keys Cell 3 derives for a site; everything else in cfg_site.json is only a record.
SITE_KEYS = ("utc_offset", "date_start", "date_end", "eco_possible", "x_days")


def load_site_cfg():
    """Restore the site-derived settings from cfg_site.json (not the whole snapshot)."""
    CFG.update({k: v for k, v in json.load(open(P("cfg_site.json"))).items() if k in SITE_KEYS})


def use_site(site, quiet=False):
    global SITE, WORK, ARCHIVE, CACHE_HLS, CACHE_HLS3, CACHE_ECO, CACHE_LST, CACHE_WX, CACHE_S1, CACHE_PS, FIG_DIR
    SITE       = site
    WORK       = os.path.join(CFG["work_root"], site)
    ARCHIVE    = os.path.join(CFG["archive_root"], site)
    CACHE_HLS  = os.path.join(WORK, "cache", "hls_v2")   # v8: NDVI/NDTI only (kept for its cloud-skip markers)
    CACHE_HLS3 = os.path.join(WORK, "cache", "hls_v3")   # v9: six reflectance bands per scene
    CACHE_ECO  = os.path.join(WORK, "cache", "eco")
    CACHE_LST  = os.path.join(WORK, "cache", "lst")      # Landsat C2 surface temperature
    CACHE_WX   = os.path.join(WORK, "cache", "daymet")   # v10: Daymet daily weather per location
    CACHE_S1   = os.path.join(WORK, "cache", "s1")       # v11: Sentinel-1 RTC per scene
    CACHE_PS   = os.path.join(WORK, "cache", "ps")       # v11: PlanetScope 8-band SR per scene
    FIG_DIR    = os.path.join(WORK, "figures")
    for d in (WORK, CACHE_HLS, CACHE_HLS3, CACHE_ECO, CACHE_LST, CACHE_WX, CACHE_S1, CACHE_PS, FIG_DIR):
        os.makedirs(d, exist_ok=True)
    if os.path.isfile(P("cfg_site.json")):
        load_site_cfg()
    if not quiet:
        print(f"\n{'─'*12} {site} {'─'*12}")


def have(name):
    """True when a product exists in the current site's working dir."""
    p = P(name)
    return os.path.isfile(p) and os.path.getsize(p) > 0


def load_da(path, name=None):
    """Open a NetCDF written by this notebook and return it as a DataArray.

    Reads fully into memory and CLOSES the file (a lazily-open NetCDF keeps an
    HDF5 lock that turns a later rewrite into a bare PermissionError), selects
    the variable by name (write_crs() adds a spatial_ref variable, so a guessing
    reader would refuse), and restores the CRS that to_netcdf demoted to an
    attribute.
    """
    wkt = None
    with xr.open_dataset(path) as ds:
        if name is None:
            name = next(v for v in ds.data_vars if v != "spatial_ref")
        da = ds[name].load()
        if da.rio.crs is None and "spatial_ref" in ds:
            att = ds["spatial_ref"].attrs
            wkt = att.get("crs_wkt") or att.get("spatial_ref")
    return da.rio.write_crs(wkt) if wkt else da


def mirror(local_path):
    """Copy one freshly written cache file to the current site's Data Store
    folder (sequential copy — the one write pattern the FUSE mount handles
    well). Returns the mirror path."""
    rel = os.path.relpath(local_path, WORK)
    dst = os.path.join(ARCHIVE, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(local_path, dst)
    return dst


# --- raster and calendar helpers (Cells 12-17) -------------------------------
LAYERS = ["n_green", "ndvi_int", "up_slope", "term_drop", "peak_win", "spring_rel"]


def write_tif(path, arr, grid, tags=None):
    """One single-band GeoTIFF on `grid` (float32 unless the array is integer).
    `tags` (e.g. the detection window) are stored in the file's metadata."""
    dt = "int32" if np.issubdtype(np.asarray(arr).dtype, np.integer) else "float32"
    (xr.DataArray(np.asarray(arr).astype(dt), coords={"y": grid.y.values, "x": grid.x.values},
                  dims=("y", "x"))
       .rio.write_crs(grid.rio.crs).rio.write_nodata(None)
       .rio.to_raster(path, compress="DEFLATE", tags=tags or {}))


def read_tif(path):
    with rasterio.open(path) as src:
        return src.read(1)


def build_season_windows(labels):
    """One detection window per season from the cover-crop record (see CFG["windows"])."""
    R, rows = CFG["windows"], []
    for season, g in labels.groupby("season"):
        y0 = int(season[:4]); y1 = y0 + 1
        ts = lambda s: pd.Timestamp(s.replace("y0", str(y0)).replace("y1", str(y1)))
        cc = g[g.cc_present == 1]
        r = (cc if len(cc) else g).iloc[0]
        method = str(r.get("cc_seeding_method", "") or "")
        seed = pd.to_datetime(r.get("cc_seed_date"), errors="coerce")
        term = pd.to_datetime(r.get("cc_termination_date"), errors="coerce")
        end_src = "paper termination"
        if pd.isna(term):
            term, end_src = ts(R["default_term"]), "default (not reported)"
        if "interseed" in method.lower() and pd.notna(seed):
            start, start_src = seed, "paper interseeding date"
        else:
            start = ts(R["fall_start"])
            start_src = "spring after fall seeding" + ("" if pd.notna(seed) else " (seed date not reported)")
        rows.append(dict(season=season, window_start=start.date(), window_end=term.date(),
                         days=(term - start).days + 1, cc_seed_date=None if pd.isna(seed) else seed.date(),
                         termination=term.date(), seeding_method=method or "not reported",
                         start_source=start_src, end_source=end_src))
    return pd.DataFrame(rows)


def season_window(season):
    """Dates of one season's detection window and its sub-windows (Cells 13-17).

    win    detection window: n_green, ndvi_int, peak_win
    rise   last rise_days before termination (inside win): up_slope
    termw  termination − term_before … + term_after: term_drop
    late   last late_days before termination: spring_rel, et_spring
    """
    R = CFG["windows"]
    w = pd.read_csv(P("season_windows.csv")).set_index("season").loc[season]
    start, end = pd.Timestamp(w.window_start), pd.Timestamp(w.window_end)
    day = lambda n: pd.Timedelta(days=n)
    return dict(season=season, y0=int(season[:4]), y1=int(season[:4]) + 1,
                start=start, end=end, win=(start, end),
                seed=pd.to_datetime(w.cc_seed_date), term=end,
                rise=(max(start, end - day(R["rise_days"])), end),
                termw=(end - day(R["term_before"]), end + day(R["term_after"])),
                late=(max(start, end - day(R["late_days"])), end),
                label=f"{start:%Y-%m-%d} → {end:%Y-%m-%d}")


# --- shared by the HLS (Cell 6) and ECOSTRESS (Cell 9) downloads ------------
import threading
GDAL_OPTS = {}                                 # filled by Cell 2 after login
NC_LOCK   = threading.Lock()                   # the netCDF/HDF5 library is not thread-safe
N_WORKERS = 8                                  # scenes fetched at once; the wait is latency


def open_clip(url, roi, masked=True):
    """Read ONLY the ROI window of a (remote) cloud-optimised GeoTIFF, in the
    file's own CRS, and return it as a georeferenced DataArray.

    Plain rasterio window read on purpose: rioxarray's clip on a lazily opened
    remote file pulls the whole 3660 x 3660 band before clipping, which turns
    a 100 KB read into a 13 MB one. GDAL does the HTTPS range requests itself
    (bearer token from Cell 2), so this is safe to run from several threads."""
    with rasterio.open(url) as src:
        bb = roi.to_crs(src.crs).total_bounds
        win = (rasterio.windows.from_bounds(*bb, transform=src.transform)
               .round_offsets().round_lengths()
               .intersection(rasterio.windows.Window(0, 0, src.width, src.height)))
        arr = src.read(1, window=win, masked=masked)
        tr, crs = src.window_transform(win), src.crs
    if masked:                       # int16 reflectance: cast BEFORE filling with NaN
        arr = arr.astype("float32").filled(np.nan)
    ny, nx = arr.shape
    da = xr.DataArray(arr, dims=("y", "x"),
                      coords={"y": tr.f + tr.e * (np.arange(ny) + 0.5),
                              "x": tr.c + tr.a * (np.arange(nx) + 0.5)})
    return da.rio.write_crs(crs).rio.write_transform(tr)


# --- CMR by tile name (shared by Cells 5 and 8) ------------------------------
# CMR's bounding-box search on HLS/ECOSTRESS is slow and silently incomplete;
# listing a tile's granules by name is fast and exact.
CMR_URL = "https://cmr.earthdata.nasa.gov/search/granules.umm_json"


def mgrs_square(lon, lat, epsg):
    """MGRS 100 km square of a point in UTM zone `epsg` (e.g. '16TFL'), with
    the square's lower-left corner in 100 km units. The latitude band comes
    from the square's centre, as the Sentinel-2 / HLS tiling does."""
    from pyproj import Transformer
    zone = int(str(epsg)[-2:])
    fwd = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
    inv = Transformer.from_crs(f"EPSG:{epsg}", "EPSG:4326", always_xy=True)
    x, y = fwd.transform(lon, lat)
    e, n = int(x // 100000), int(y // 100000)
    _, lat_c = inv.transform((e + 0.5) * 100000, (n + 0.5) * 100000)
    band = "CDEFGHJKLMNPQRSTUVWX"[min(int((lat_c + 80) // 8), 19)]
    col = "ABCDEFGHJKLMNPQRSTUVWXYZ"[(zone - 1) % 3 * 8 + e - 1]
    row = "ABCDEFGHJKLMNPQRSTUV"[(n + (5 if zone % 2 == 0 else 0)) % 20]
    return f"{zone:02d}{band}{col}{row}", e, n


def roi_tiles(roi, utm):
    """MGRS tiles that cover the whole ROI (a 109.8 km HLS/ECOSTRESS tile is
    anchored at its 100 km square's north-west corner). Falls back to every
    tile touched by a corner when none covers the ROI on its own."""
    b = roi.total_bounds
    squares = {mgrs_square(lon, lat, utm.to_epsg())
               for lon, lat in [(b[0], b[1]), (b[2], b[1]), (b[2], b[3]), (b[0], b[3])]}
    x0, y0, x1, y1 = roi.to_crs(utm).total_bounds
    full = [t for t, e, n in sorted(squares)
            if e * 1e5 <= x0 and x1 <= e * 1e5 + 109800
            and (n + 1) * 1e5 - 109800 <= y0 and y1 <= (n + 1) * 1e5]
    return full or [t for t, _, _ in sorted(squares)]


def cmr_by_name(short_name, pattern, page_size=2000, timeout=120):
    """Every granule of `short_name` whose name matches `pattern` (CMR
    search-after paging). Asserts the count against CMR's own hit count."""
    items, headers = [], {}
    params = {"short_name": short_name, "readable_granule_name[]": pattern,
              "options[readable_granule_name][pattern]": "true", "page_size": page_size}
    while True:
        r = requests.get(CMR_URL, params=params, headers=headers, timeout=timeout)
        r.raise_for_status()
        page = r.json()["items"]
        items += page
        after = r.headers.get("CMR-Search-After")
        if not after or len(page) < page_size:
            break
        headers["CMR-Search-After"] = after
    hits = int(r.headers.get("CMR-Hits", len(items)))
    assert len(items) == hits, f"{pattern}: listed {len(items)} of {hits}"
    return items


# --- restore every site from its archive -----------------------------------
for SITE in SITES:
    use_site(SITE, quiet=True)
    RESTORED = []
    if CFG["use_archive"] and os.path.isdir(ARCHIVE):
        for root, _, files in os.walk(ARCHIVE):
            rel = os.path.relpath(root, ARCHIVE)
            for f in sorted(files):
                dst = P(f) if rel == "." else P(os.path.join(rel, f))
                if not os.path.exists(dst):
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    shutil.copy2(os.path.join(root, f), dst)
                    RESTORED.append(os.path.relpath(dst, WORK))
    top = [f for f in RESTORED if os.sep not in f]
    print(f"{SITE:<18} restored from archive: {len(top)} product(s), "
          f"{len(RESTORED) - len(top)} cached scene(s)")

# Prove the working disk really accepts a NetCDF write.
_probe = os.path.join(CFG["work_root"], ".write_test.nc")
if os.path.exists(_probe):
    os.remove(_probe)
xr.DataArray(np.zeros((2, 2), "float32"), dims=("y", "x")).to_netcdf(_probe)
os.remove(_probe)
print(f"\nsites       : {', '.join(SITES)}")
print(f"working dir : {CFG['work_root']}/<site>")
print(f"archive     : {CFG['archive_root']}/<site>")
print("NetCDF write test: OK")

## Cell 2 — Earthdata authentication

`persist=True` writes `~/.netrc` on first use. When the archive already holds
every download product and no login is available, the cell records
`auth = None` and the download cells skip.

In [ ]:
# =============================================================================
# Cell 2 — Authenticate
# =============================================================================
try:
    auth = earthaccess.login(persist=True)
    assert auth.authenticated
    # GDAL reads the HLS GeoTIFFs directly over HTTPS with this bearer token
    # (Cell 6); no per-file Python handles, and thread-safe.
    # (GDAL_HTTP_HEADERS is ignored by some builds; a header file always works.)
    _hdr = os.path.expanduser("~/.edl_gdal_header")
    with open(_hdr, "w") as _f:
        _f.write(f"Authorization: Bearer {auth.token['access_token']}\n")
    os.chmod(_hdr, 0o600)
    GDAL_OPTS = dict(GDAL_HTTP_HEADER_FILE=_hdr,
                     GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
                     CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif,.tiff",
                     GDAL_HTTP_MAX_RETRY="4", GDAL_HTTP_RETRY_DELAY="3")
    print("Earthdata: authenticated (bearer token handed to GDAL)")
except Exception as e:
    if have("hls_scenes.nc") and have("fused_cube.nc"):
        auth, GDAL_OPTS = None, {}
        print(f"Earthdata login unavailable ({type(e).__name__}) — continuing "
              f"with archived products; nothing will be downloaded")
    else:
        raise RuntimeError("Earthdata login failed and no archived products "
                           "to fall back on — create ~/.netrc or log in") from e

## Cell 3 — Site, labels, ROI, period, windows

Everything here comes out of the ground-truth store. Only labels with
`usable_for_rs` and a known `cc_present` (0/1) are kept; −1 rows are unknown
and never enter an evaluation.

Windows are half-open `[start, next)` and tile the period with no gap. The
window length is 8 days when the period starts after S30 came online, 16 days
otherwise.

In [ ]:
# =============================================================================
# Cell 3 — Load the site's fields and labels; derive ROI, period, windows
# =============================================================================
def make_windows(start, end, split, x_before, x_after):
    """Consecutive HALF-OPEN windows tiling [start, end]: `x_before` days long
    before `split` (Landsat only), `x_after` days from `split` on (L30 + S30).
    A window never straddles `split`."""
    start, end, split = pd.Timestamp(start), pd.Timestamp(end), pd.Timestamp(split)
    end_excl = end + pd.Timedelta(days=1)
    edges = []
    for seg_start, seg_end, x in ((start, min(split, end_excl), x_before),
                                  (max(split, start), end_excl, x_after)):
        if seg_start >= seg_end:
            continue
        e = pd.date_range(seg_start, seg_end - pd.Timedelta(days=1), freq=f"{x}D")
        edges += [(ei, min(ei + pd.Timedelta(days=x), seg_end)) for ei in e]
    w = pd.DataFrame([dict(win_id=i, win_start=s, win_next=n, win_end=n - pd.Timedelta(days=1),
                           win_center=s + (n - s) / 2, win_days=(n - s).days)
                      for i, (s, n) in enumerate(edges)])
    assert (w.win_next.values[:-1] == w.win_start.values[1:]).all()   # no gaps, no overlap
    assert w.win_start.iloc[0] == start and w.win_next.iloc[-1] == end_excl
    return w


for SITE in SITES:
    use_site(SITE)
    fields_all  = gpd.read_file(CFG["gt_gpkg"], layer="fields")
    seasons_all = pd.DataFrame(gpd.read_file(CFG["gt_gpkg"], layer="cc_seasons"))
    sites_all   = gpd.read_file(CFG["gt_gpkg"], layer="sites")

    fields = fields_all[fields_all.site_id == SITE].reset_index(drop=True)
    assert len(fields), f"no fields for site {SITE} — check CFG['site']"
    seasons = seasons_all[seasons_all.site_id == SITE].copy()
    seasons["usable"] = seasons.usable_for_rs.astype(str).str.lower().isin(["true", "1"])
    labels = seasons[seasons.usable & (seasons.cc_present >= 0)].copy()
    keep_seasons = (CFG["seasons"] or {}).get(SITE)
    if keep_seasons:
        labels = labels[labels.season.isin(keep_seasons)]
    for c in ("detect_start", "detect_end", "best_start", "best_end",
              "cc_seed_date", "cc_termination_date", "next_cash_plant"):
        labels[c] = pd.to_datetime(labels[c], errors="coerce")
    assert len(labels), f"no usable labels for {SITE}"

    # --- ROI: fields' bounding box grown by the buffer, in a metric CRS ---------
    utm = fields.estimate_utm_crs()
    b = fields.to_crs(utm).total_bounds
    roi = gpd.GeoDataFrame({"name": [SITE]},
                           geometry=[box(b[0] - CFG["roi_buffer_m"], b[1] - CFG["roi_buffer_m"],
                                         b[2] + CFG["roi_buffer_m"], b[3] + CFG["roi_buffer_m"])],
                           crs=utm).to_crs("EPSG:4326")
    roi.to_file(P("roi.geojson"), driver="GeoJSON")
    fields.to_file(P("fields_site.gpkg"), driver="GPKG")
    labels.to_csv(P("seasons_site.csv"), index=False)
    CFG["utc_offset"] = int(round(roi.geometry.centroid.x.iloc[0] / 15))

    # --- detection window of every season, from the cover-crop record ----------
    sw = build_season_windows(labels)
    sw.to_csv(P("season_windows.csv"), index=False)

    # --- period and window length from the labels ------------------------------
    # early enough for the first detection window (an interseeding date can be
    # in June), late enough to see the cash crop emerge after termination
    date_start = (min(labels.detect_start.min(), pd.Timestamp(sw.window_start.min()))
                  - pd.Timedelta(days=CFG["lead_days"]))
    date_end   = (max(labels.detect_end.max(), pd.Timestamp(sw.window_end.max()))
                  + pd.Timedelta(days=CFG["tail_days"]))
    CFG.update(date_start=str(date_start.date()), date_end=str(date_end.date()),
               eco_possible=bool(date_end >= pd.Timestamp(CFG["eco_start"])))




    windows = make_windows(CFG["date_start"], CFG["date_end"], CFG["s30_start"],
                           CFG["x_days_1sat"], CFG["x_days_4sat"])
    x_days = int(windows.win_days.mode().iloc[0])            # dominant length, for labels
    CFG["x_days"] = x_days
    windows.to_csv(P("windows.csv"), index=False)
    json.dump({k: v for k, v in CFG.items()}, open(P("cfg_site.json"), "w"), indent=2, default=str)

    w_km = (b[2] - b[0] + 2 * CFG["roi_buffer_m"]) / 1000
    h_km = (b[3] - b[1] + 2 * CFG["roi_buffer_m"]) / 1000
    print(f"site     : {SITE}   fields {len(fields)}  "
          f"({', '.join(sorted(fields.role.unique()))})")
    print(f"labels   : {len(labels)} usable field-seasons  "
          f"({int((labels.cc_present == 1).sum())} CC / {int((labels.cc_present == 0).sum())} no CC)"
          f"  seasons {sorted(labels.season.unique())}")
    print(f"ROI      : {w_km:.1f} x {h_km:.1f} km   ({utm.name})")
    print(f"period   : {CFG['date_start']} → {CFG['date_end']}   {len(windows)} windows "
          + " + ".join(f"{n} x {d} d" for d, n in windows.win_days.value_counts().sort_index(ascending=False).items()))
    print(f"ECOSTRESS: {'possible' if CFG['eco_possible'] else 'not available for this period'}")
    print(f"\ndetection window per season (same dates for every field of the season):")
    for _, w in sw.iterrows():
        print(f"  {w.season}  {w.window_start} → {w.window_end}  ({w.days:>3} d)  "
              f"{w.seeding_method[:34]:<34}  start: {w.start_source}; end: {w.end_source}")
    print(f"saved    : roi.geojson, fields_site.gpkg, seasons_site.csv, season_windows.csv, windows.csv, cfg_site.json")

## Cell 4 — Reference grids

Built from the ROI polygon, never from whichever scene arrives first (a
scene-derived grid inherits that scene's footprint and silently truncates
everything else). Both grids share one CRS and origin.

In [ ]:
# =============================================================================
# Cell 4 — Build the 30 m and 70 m reference grids from the ROI
# =============================================================================
def make_grid(roi, res, crs):
    """An empty raster spanning the ROI at `res` metres, derived from the ROI."""
    x0, y0, x1, y1 = roi.to_crs(crs).total_bounds
    nx, ny = int(np.ceil((x1 - x0) / res)), int(np.ceil((y1 - y0) / res))
    return xr.DataArray(
        np.full((ny, nx), np.nan, "float32"),
        coords={"y": y1 - (np.arange(ny) + 0.5) * res,
                "x": x0 + (np.arange(nx) + 0.5) * res},
        dims=("y", "x"), name="grid").rio.write_crs(crs)


for SITE in SITES:
    use_site(SITE)
    roi = gpd.read_file(P("roi.geojson"))
    utm = roi.estimate_utm_crs()




    grid30 = make_grid(roi, CFG["res_optical"], utm)
    grid70 = make_grid(roi, CFG["res_thermal"], utm)
    # netCDF4 refuses to overwrite a file another process holds open; unlink first.
    for path in (P("grid_30m.nc"), P("grid_70m.nc")):
        if os.path.exists(path):
            os.remove(path)
    grid30.to_netcdf(P("grid_30m.nc"))
    grid70.to_netcdf(P("grid_70m.nc"))

    rx0, ry0, rx1, ry1 = roi.to_crs(utm).total_bounds
    for g, res in ((grid30, CFG["res_optical"]), (grid70, CFG["res_thermal"])):
        assert g.sizes["x"] * res >= (rx1 - rx0) and g.sizes["y"] * res >= (ry1 - ry0)
        print(f"{res:>3} m grid: {g.sizes['y']:>4} x {g.sizes['x']:<4} px")
    print(f"CRS: {utm.name}\nsaved: grid_30m.nc, grid_70m.nc")

## Cell 5 — Find HLS scenes (now with blue and green)

`HLSL30` (Landsat 8/9) and `HLSS30` (Sentinel-2 A/B) are already harmonised by
NASA — one 30 m grid, one atmospheric correction, BRDF-normalised, with
Sentinel-2 bands adjusted to Landsat band-passes. Combining them into a single
series is the entire purpose of the product.

**Search by tile name, not by bounding box.** CMR's spatial search on the HLS
collections is slow (a minute per request on a bad day) and, worse, returns
an incomplete list without saying so — a first run of this notebook got a
third of the scenes that exist. The MGRS tile that covers a small ROI is
computed from its coordinates instead, and every granule of that tile is
listed by name (`HLS.S30.T16TFL.2019*`), which CMR answers in a fraction of a
second and completely; the cell asserts that the count matches CMR's own
`CMR-Hits`.

Two details that fail without an error message:

- **`HLSF30` does not exist.** CMR ignores an unknown short name.
- **NIR differs by sensor**: Landsat `B05`, Sentinel-2 **`B8A`** (narrow NIR,
  865 nm) — *not* `B08`. NASA harmonised against B8A; B08 introduces a bias.

Scene cloud cover lives in `AdditionalAttributes.CLOUD_COVERAGE`. Missing
metadata is treated as 0% (keep the scene) — per-pixel Fmask does the real
masking.

v9 also resolves the blue (B02) and green (B03) bands of both sensors: the Ma et al.
baseline uses all six reflectance bands and GCVI needs green.

In [ ]:
# =============================================================================
# Cell 5 — List every HLS granule of the ROI's tile, resolve band URLs, save the index
# =============================================================================
def hls_row(g):
    """One index row from a CMR UMM granule record; None if a band is missing."""
    parts = g["meta"]["native-id"].split(".")         # HLS.S30.T16TFL.2024274T16...
    sensor, tile = parts[1], parts[2]
    attrs = {a.get("Name", ""): a.get("Values", [None])[0]
             for a in g["umm"].get("AdditionalAttributes", [])}
    try:
        cloud = float(attrs.get("CLOUD_COVERAGE"))
    except (TypeError, ValueError):
        cloud = np.nan                                # unknown ≠ cloudy
    links = [u["URL"] for u in g["umm"].get("RelatedUrls", [])
             if u.get("Type") == "GET DATA" and u["URL"].startswith("https")]
    pick = lambda s: next((l for l in links if l.endswith(f".{s}.tif")), None)
    bands = {k: pick(v) for k, v in HLS_BANDS[sensor].items()}
    fmask = pick("Fmask")
    if not all(bands.values()) or not fmask:
        return None
    return dict(gid=g["meta"]["native-id"], sensor=sensor, tile=tile,
                date=pd.to_datetime(parts[3][:7], format="%Y%j"),
                cloud=cloud, fmask=fmask, **bands)


for SITE in SITES:
    use_site(SITE)
    load_site_cfg()     # period from Cell 3
    roi = gpd.read_file(P("roi.geojson"))
    utm = roi.estimate_utm_crs()

    # NIR and SWIR band ids differ by sensor. Getting one wrong raises nothing --
    # it just yields an index with no physical meaning.
    HLS_BANDS = {"L30": {"blue": "B02", "green": "B03", "red": "B04",
                         "nir": "B05",                   # Landsat 8/9
                         "swir1": "B06", "swir2": "B07"},
                 "S30": {"blue": "B02", "green": "B03", "red": "B04",
                         "nir": "B8A",                   # Sentinel-2: B8A, not B08
                         "swir1": "B11", "swir2": "B12"}}




    old_index = have("hls_index.csv") and "blue" not in pd.read_csv(P("hls_index.csv"), nrows=1).columns
    if globals().get("auth") is None and have("hls_index.csv") and not old_index:
        hls_index = pd.read_csv(P("hls_index.csv"), parse_dates=["date"])
        print(f"Earthdata offline — using archived hls_index.csv "
              f"({len(hls_index)} scenes, {hls_index.date.min().date()} → "
              f"{hls_index.date.max().date()})")
    else:
        tiles = roi_tiles(roi, utm)
        tile = tiles[0]                                   # any full-cover tile has all the pixels
        t_start, t_end = pd.Timestamp(CFG["date_start"]), pd.Timestamp(CFG["date_end"])
        rows, n_found, n_cloudy = [], 0, 0
        for sensor in ("L30", "S30"):
            for year in range(t_start.year, t_end.year + 1):
                for g in cmr_by_name(f"HLS{sensor}", f"HLS.{sensor}.T{tile}.{year}*"):
                    n_found += 1
                    r = hls_row(g)
                    if r is None or not (t_start <= r["date"] <= t_end):
                        continue
                    if np.nan_to_num(r["cloud"], nan=0.0) > CFG["cloud_max"]:
                        n_cloudy += 1
                        continue
                    rows.append(r)
        hls_index = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
        assert len(hls_index), "no usable HLS granule — check the tile and period"
        hls_index.to_csv(P("hls_index.csv"), index=False)

        print(f"tile           : {tile}  (ROI fully inside; candidates {tiles})")
        print(f"granules       : {n_found} listed for the tile  →  in period and ≤ "
              f"{CFG['cloud_max']:.0f}% cloud: {len(hls_index)}  (cloudier: {n_cloudy})")
        print(f"sensors        : {hls_index.sensor.value_counts().to_dict()}")
        print(f"dates          : {hls_index.date.min().date()} → {hls_index.date.max().date()}")
        print(f"saved          : hls_index.csv")

## Cell 6 — Download the six reflectance bands

Same windowed, threaded, Fmask-first reader as v8, but the cache now keeps the
six bands (blue, green, red, NIR, SWIR1, SWIR2) as int16 reflectance × 10 000
in `cache/hls_v3`, so any index can be derived later without downloading
again. Scenes the v8 cache already marked as clouded out are not fetched.

In [ ]:
# =============================================================================
# Cell 6 — Download the six reflectance bands of each scene, cache as NetCDF (resumable)
# =============================================================================
def norm_diff(x, y):
    """(x-y)/(x+y), clipped to the physically possible range."""
    v = (x - y) / (x + y)
    return v.where((v >= -1) & (v <= 1))

BANDS6 = ["blue", "green", "red", "nir", "swir1", "swir2"]


def fetch_scene(r, cache_dir, grid, roi, max_retry=3):
    """One scene: Fmask first; the six bands only if enough of the ROI is clear.

    Stores raw HLS reflectance (DN = reflectance × 10 000) as int16 with
    −9999 for cloud/shadow/snow/water/no-data; NDVI, NDTI, GCVI … are derived
    later (Cell 7, Cell 19), so a new index never needs a new download.

    Returns (status, error): "hit" (cached, or clouded out in the v8 cache
    too), "skipped" (clouded out — a .skip marker is written), "new", "failed".
    """
    path = os.path.join(cache_dir, r.gid + ".nc")        # granule id = unique
    skip = path[:-3] + ".skip"
    old_skip = os.path.join(CACHE_HLS, r.gid + ".skip")  # v8 already saw it clouded out
    if os.path.exists(path) or os.path.exists(skip) or os.path.exists(old_skip):
        return "hit", None
    err = None
    for attempt in range(max_retry):
        try:
            with rasterio.Env(**GDAL_OPTS):
                fmask = open_clip(r.fmask, roi, masked=False).rio.reproject_match(grid)
                clear = (np.nan_to_num(fmask.values, nan=255).astype("uint8") & FMASK_BAD) == 0
                if clear.mean() < MIN_CLEAR:
                    with open(skip, "w") as f:
                        f.write(f"clear fraction {clear.mean():.3f}\n")
                    mirror(skip)
                    return "skipped", None
                # Always reproject onto the reference grid. Comparing shapes alone
                # is unsafe: two MGRS tiles can clip to identical shapes with
                # different origins, and that misalignment is invisible once stored.
                ds = xr.Dataset()
                for b in BANDS6:
                    da = open_clip(r[b], roi).rio.reproject_match(grid)
                    ds[b] = da.where(xr.DataArray(clear, coords=da.coords, dims=da.dims))
                    # no inherited scale_factor (it would be re-applied on read);
                    # int16 on disk with an explicit fill value
                    ds[b].attrs = {}
                    ds[b].encoding = {"dtype": "int16", "_FillValue": -9999, "zlib": True}
                ds.attrs = dict(gid=r.gid, sensor=r.sensor, date=str(r.date.date()),
                                units="HLS reflectance x 10000")
                with NC_LOCK:               # one NetCDF write at a time
                    ds.to_netcdf(path)
                mirror(path)                # Data Store copy: survives the container
                return "new", None
        except Exception as e:
            err = f"{type(e).__name__}: {e}"
            time.sleep(5 * 2 ** attempt)                   # exponential back-off
    return "failed", err

def download_hls(index, cache_dir, grid, roi):
    """Fetch every scene of `index` with N_WORKERS threads (resumable)."""
    from concurrent.futures import ThreadPoolExecutor, as_completed
    t0 = time.time()
    counts = dict(new=0, hit=0, skipped=0, failed=0)
    failed = []
    rows = [r for _, r in index.iterrows()]
    ex = ThreadPoolExecutor(N_WORKERS)
    futs = {ex.submit(fetch_scene, r, cache_dir, grid, roi): r.gid for r in rows}
    try:
        for k, f in enumerate(as_completed(futs), 1):
            status, err = f.result()
            counts[status] += 1
            if err:
                failed.append(dict(gid=futs[f], error=err[:80]))
            fetched = counts["new"] + counts["skipped"] + counts["failed"]
            if fetched in (12, 50) or (fetched and fetched % 200 == 0):
                rate = fetched / (time.time() - t0)
                print(f"  {k}/{len(rows)}  new {counts['new']}  cloud-skipped "
                      f"{counts['skipped']}  ({(time.time()-t0)/60:.1f} min, "
                      f"~{(len(rows)-k)/rate/60:.0f} min left)")
    except KeyboardInterrupt:
        # Stop the queue, not just the main thread; scenes in flight finish.
        ex.shutdown(wait=False, cancel_futures=True)
        print(f"interrupted after {counts['new']} new scenes — re-run to resume")
        raise
    ex.shutdown(wait=True)
    print(f"\nHLS: {counts['new']} new, {counts['skipped']} cloud-skipped, "
          f"{counts['hit']} cached, {counts['failed']} failed "
          f"({(time.time()-t0)/60:.1f} min)")
    if failed:
        print(pd.DataFrame(failed).to_string(index=False))
        print("Re-run this cell to retry only the failures.")
    return failed


for SITE in SITES:
    use_site(SITE)
    hls_index = pd.read_csv(P("hls_index.csv"), parse_dates=["date"])
    grid30 = load_da(P("grid_30m.nc")).rio.write_crs(
        gpd.read_file(P("roi.geojson")).estimate_utm_crs())
    roi = gpd.read_file(P("roi.geojson"))

    # bits 0-5 = cirrus, cloud, adjacent, shadow, snow, water. 6-7 = aerosol: keep.
    FMASK_BAD = sum(1 << b for b in range(6))         # 0b00111111 = 63
    assert (64 & FMASK_BAD) == 0 and (128 & FMASK_BAD) == 0, "aerosol bits masked!"


    MIN_CLEAR = 0.05   # ROI clear fraction below which the scene is not worth its 6 bands








    if not os.listdir(CACHE_HLS3) and have("hls_bands.nc") and globals().get("auth") is None:
        # The per-scene cache did not survive the container, but its product did.
        print("per-scene cache is empty and hls_bands.nc was restored from the "
              "archive — nothing to download.\n"
              "Set CFG['use_archive'] = False to rebuild from raw scenes.")
        hls_failed = []
    else:
        hls_failed = download_hls(hls_index, CACHE_HLS3, grid30, roi)
        n_nc = len([f for f in os.listdir(CACHE_HLS3) if f.endswith(".nc")])
        n_sk = len([f for f in os.listdir(CACHE_HLS3) if f.endswith(".skip")]) + \
               len([f for f in os.listdir(CACHE_HLS) if f.endswith(".skip")])
        print(f"cache now holds {n_nc} scenes + {n_sk} cloud-skip markers "
              f"of {len(hls_index)} indexed")

## Cell 7 — Stack the bands; derive NDVI and NDTI

`hls_bands.nc` holds the six bands per observation day (same-day L30 + S30
averaged); `hls_scenes.nc` keeps the v8 NDVI/NDTI stack for Cells 10–17.

In [ ]:
# =============================================================================
# Cell 7 — Assemble the per-scene stacks: six bands + NDVI + NDTI
# =============================================================================
BANDS6 = ["blue", "green", "red", "nir", "swir1", "swir2"]


def nd(a, b):
    """(a-b)/(a+b), NaN outside the physically possible range."""
    with np.errstate(all="ignore"):
        v = (a - b) / (a + b)
    return np.where((v >= -1) & (v <= 1), v, np.nan).astype("float32")


for SITE in SITES:
    use_site(SITE)
    gids = set(pd.read_csv(P("hls_index.csv")).gid) if have("hls_index.csv") else None
    files = sorted(f for f in os.listdir(CACHE_HLS3) if f.endswith(".nc")
                   and (gids is None or f[:-3] in gids))      # only this period's scenes

    if not files and have("hls_bands.nc") and have("hls_scenes.nc"):
        with xr.open_dataset(P("hls_bands.nc")) as _b:
            nb_ = _b.sizes["time"]
        print(f"no per-scene cache — using restored hls_bands.nc / hls_scenes.nc ({nb_} days)")
        continue
    assert files, "HLS band cache is empty — run Cell 6 first"

    stacks, dates, fracs = {b: [] for b in BANDS6}, [], []
    for fn in files:
        with xr.open_dataset(os.path.join(CACHE_HLS3, fn)) as ds:   # int16 → float, −9999 → NaN
            red = ds["red"].values
            frac = float(np.isfinite(red).mean())
            if frac == 0:                       # scene entirely clouded out
                continue
            for b in BANDS6:
                stacks[b].append((ds[b].values / 10000.0).astype("float32"))
        dates.append(pd.to_datetime(fn.split(".")[3][:7], format="%Y%j"))
        fracs.append(frac)

    grid30 = load_da(P("grid_30m.nc"))
    coords = {"time": pd.DatetimeIndex(dates), "y": grid30.y.values, "x": grid30.x.values}
    bands = xr.Dataset({b: (("time", "y", "x"), np.stack(stacks[b])) for b in BANDS6},
                       coords=coords)
    # Same-day L30 + S30 overpasses are averaged so neither is counted twice
    n_before = bands.sizes["time"]
    bands = bands.groupby("time").mean("time").sortby("time")
    for b in BANDS6:
        bands[b].attrs = {"units": "surface reflectance (0-1)"}
        bands[b].encoding = {"dtype": "int16", "scale_factor": 1e-4, "_FillValue": -9999, "zlib": True}

    # the v8 products, now derived from the bands
    scenes = xr.Dataset({"ndvi": (("time", "y", "x"), nd(bands.nir.values, bands.red.values)),
                         "ndti": (("time", "y", "x"), nd(bands.swir1.values, bands.swir2.values))},
                        coords={"time": bands.time.values, "y": bands.y.values, "x": bands.x.values})
    for name, ds_ in (("hls_bands.nc", bands), ("hls_scenes.nc", scenes)):
        if os.path.exists(P(name)):
            os.remove(P(name))                  # HDF5 locks an open file
        ds_.to_netcdf(P(name))

    fr = np.array(fracs)
    print(f"scenes loaded    : {len(files)}  →  usable {len(fracs)}")
    print(f"same-day merged  : {n_before - bands.sizes['time']}")
    print(f"observation days : {bands.sizes['time']}")
    print(f"grid             : {bands.sizes['y']} x {bands.sizes['x']} px")
    for v in ("red", "nir", "swir1"):
        a = bands[v].values
        print(f"  {v:<6} valid {np.isfinite(a).mean():.1%}  range {np.nanmin(a):.3f} … {np.nanmax(a):.3f}"
              f"   (reflectance; a maximum near 1e-4 or above 10 means a scaling error)")
    for v in ("ndvi", "ndti"):
        a = scenes[v].values
        print(f"  {v.upper():<6} valid {np.isfinite(a).mean():.1%}  range {np.nanmin(a):+.2f} … {np.nanmax(a):+.2f}")
    print(f"per-scene valid fraction: median {np.median(fr):.1%} | >90%: {(fr>0.9).sum()} | "
          f"50-90%: {((fr>0.5)&(fr<=0.9)).sum()} | <50%: {(fr<=0.5).sum()}")
    print(f"saved: hls_bands.nc (6 bands, int16 × 1e-4), hls_scenes.nc (NDVI, NDTI)")


## Cell 8 — Find ECOSTRESS scenes (only when the period allows)

ECOSTRESS began acquiring in July 2018. For a site whose labels end before
that (NE_BLANCO2017: 2013–16) there is nothing to search; the cell records the
fact and the fusion cell builds an optical-only cube.

`ECO_L3T_JET` is daily-integrated ET on the same MGRS tiles as HLS, so the
tile found in Cell 5 is listed by name here too (`ECOv002_L3T_JET_*_16TFL_2023*`)
— the spatial search is as incomplete for ECOSTRESS as it is for HLS.

In [ ]:
# =============================================================================
# Cell 8 — List the tile's ECOSTRESS granules and save the index (skipped when unavailable)
# =============================================================================
for SITE in SITES:
    use_site(SITE)
    load_site_cfg()     # period, eco_possible, …
    roi = gpd.read_file(P("roi.geojson"))
    utm = roi.estimate_utm_crs()

    ECO_PATTERNS = {"ET":  [r"ETdaily\.tif$", r"ETinst\.tif$", r"_ET\.tif$"],
                    "LST": [r"_LST\.tif$"]}
    ECO_QC = [r"_cloud\.tif$", r"_QC\.tif$"]
    ECO_COLS = ["gid", "local", "var", "qc", "hour"]

    if not CFG["eco_possible"]:
        eco_index = pd.DataFrame(columns=ECO_COLS)
        eco_index.to_csv(P("eco_index.csv"), index=False)
        print(f"ECOSTRESS skipped: period ends {CFG['date_end']}, before the first "
              f"acquisitions ({CFG['eco_start']}). Optical-only cube.")
    elif globals().get("auth") is None and have("eco_index.csv"):
        eco_index = pd.read_csv(P("eco_index.csv"), parse_dates=["local"])
        print(f"Earthdata offline — using archived eco_index.csv ({len(eco_index)} granules)")
    else:
        tile = roi_tiles(roi, utm)[0]
        t_start = max(pd.Timestamp(CFG["date_start"]), pd.Timestamp(CFG["eco_start"]))
        t_end = pd.Timestamp(CFG["date_end"])
        rows, n_found = [], 0
        for year in range(t_start.year, t_end.year + 1):
            for g in cmr_by_name(CFG["eco_product"], f"ECOv002_L3T_JET_*_{tile}_{year}*"):
                n_found += 1
                links = [u["URL"] for u in g["umm"].get("RelatedUrls", [])
                         if u.get("Type") == "GET DATA" and u["URL"].startswith("https")]
                var_url = next((l for p in ECO_PATTERNS[CFG["eco_var"]]
                                for l in links if re.search(p, l)), None)
                if not var_url:
                    continue
                qc_url = next((l for p in ECO_QC for l in links if re.search(p, l)), None)
                try:
                    t = pd.to_datetime(g["umm"]["TemporalExtent"]["RangeDateTime"]
                                       ["BeginningDateTime"]).tz_localize(None)
                except Exception:
                    continue
                if not (t_start <= t <= t_end + pd.Timedelta(days=1)):
                    continue
                rows.append(dict(gid=g["meta"]["native-id"],
                                 local=t + pd.Timedelta(hours=CFG["utc_offset"]),
                                 var=var_url, qc=qc_url))
        eco_index = pd.DataFrame(rows, columns=ECO_COLS[:-1])
        if len(eco_index):
            eco_index = eco_index.sort_values("local").reset_index(drop=True)
            eco_index["hour"] = eco_index.local.dt.hour + eco_index.local.dt.minute / 60
        else:
            eco_index["hour"] = []
        eco_index.to_csv(P("eco_index.csv"), index=False)
        print(f"{CFG['eco_product']} / {CFG['eco_var']}: tile {tile}, {n_found} granules listed "
              f"→ {len(eco_index)} in {str(t_start.date())} → {CFG['date_end']}")
        if len(eco_index):
            print(f"  overpass hours : {eco_index.hour.min():.1f} – {eco_index.hour.max():.1f} local")
            print(f"  with QC layer  : {eco_index['qc'].notna().sum()}")
        print("  saved          : eco_index.csv")

## Cell 9 — Download ECOSTRESS (skipped when there is nothing to download)

Each granule's ROI window is read (same windowed GDAL read and thread pool as
Cell 6), QC-masked, reprojected onto the 70 m grid unconditionally
(shape-matching alone lets a differently-originated tile through), cached as
NetCDF and mirrored to the Data Store.

In [ ]:
# =============================================================================
# Cell 9 — Download ECOSTRESS onto the reference grid (resumable, mirrored)
# =============================================================================
def fetch_eco(r, cache_dir, grid, roi, var, max_retry=3):
    """One ECOSTRESS granule: ROI window of the variable and its QC layer,
    QC-masked, on the 70 m grid, cached, mirrored. Same windowed GDAL read and
    thread rules as the HLS fetch (Cell 6)."""
    path = os.path.join(cache_dir, f"{var}_{r.gid}.nc")
    if os.path.exists(path):
        return "hit", None
    err = None
    for attempt in range(max_retry):
        try:
            with rasterio.Env(**GDAL_OPTS):
                val = open_clip(r["var"], roi).rio.reproject_match(grid)
                if isinstance(r.qc, str):
                    qc = open_clip(r.qc, roi, masked=False).rio.reproject_match(grid)
                    bad = np.nan_to_num(qc.values, nan=1) > 0
                    val = val.where(~xr.DataArray(bad, coords=val.coords, dims=val.dims))
                val.name = var
                val.attrs = dict(gid=r.gid, local_time=str(r.local))   # no inherited scale_factor
                val.encoding = {}
                with NC_LOCK:
                    val.to_netcdf(path)
                mirror(path)
                return "new", None
        except Exception as e:
            err = f"{type(e).__name__}: {e}"
            time.sleep(5 * 2 ** attempt)
    return "failed", err

def download_eco(index, cache_dir, grid, roi, var):
    """Fetch every granule of `index` with N_WORKERS threads (resumable)."""
    from concurrent.futures import ThreadPoolExecutor, as_completed
    t0 = time.time()
    counts = dict(new=0, hit=0, failed=0)
    failed = []
    rows = [r for _, r in index.iterrows()]
    ex = ThreadPoolExecutor(N_WORKERS)
    futs = {ex.submit(fetch_eco, r, cache_dir, grid, roi, var): r.gid for r in rows}
    try:
        for k, f in enumerate(as_completed(futs), 1):
            status, err = f.result()
            counts[status] += 1
            if err:
                failed.append(dict(gid=futs[f], error=err[:80]))
            fetched = counts["new"] + counts["failed"]
            if fetched in (12, 50) or (fetched and fetched % 200 == 0):
                rate = fetched / (time.time() - t0)
                print(f"  {k}/{len(rows)}  new {counts['new']}  ({(time.time()-t0)/60:.1f} min, "
                      f"~{(len(rows)-k)/rate/60:.0f} min left)")
    except KeyboardInterrupt:
        ex.shutdown(wait=False, cancel_futures=True)
        print(f"interrupted after {counts['new']} new granules — re-run to resume")
        raise
    ex.shutdown(wait=True)
    print(f"\n{var}: {counts['new']} new, {counts['hit']} cached, {counts['failed']} failed "
          f"({(time.time()-t0)/60:.1f} min)")
    if failed:
        print(pd.DataFrame(failed).to_string(index=False))
        print("Re-run this cell to retry only the failures.")
    return failed


for SITE in SITES:
    use_site(SITE)
    eco_index = pd.read_csv(P("eco_index.csv"), parse_dates=["local"])
    roi = gpd.read_file(P("roi.geojson"))
    grid70 = load_da(P("grid_70m.nc")).rio.write_crs(roi.estimate_utm_crs())






    if not len(eco_index):
        print("no ECOSTRESS granules for this site/period — nothing to download")
    elif globals().get("auth") is None and not os.listdir(CACHE_ECO) and have("fused_cube.nc"):
        print("offline and fused_cube.nc restored — skipping ECOSTRESS download")
    else:
        download_eco(eco_index, CACHE_ECO, grid70, roi, CFG["eco_var"])
        fr = []
        for fn in sorted(os.listdir(CACHE_ECO))[:60]:
            if fn.endswith(".nc"):
                fr.append(float(np.isfinite(load_da(os.path.join(CACHE_ECO, fn)).values).mean()))
        if fr:
            fr = np.array(fr)
            print(f"cached granules sampled: {len(fr)} — near-full (>95%): {(fr >= 0.95).sum()}, "
                  f"best coverage {fr.max():.1%} (must approach 100%)")

## Cell 10 — Fuse into an evenly-spaced cube

NDVI and NDTI: **median** per window (a cloud Fmask missed is an outlier).
ET: **mean** per window, kept on its native 70 m grid, only when scenes exist.
Gaps are filled by linear interpolation along time; the share is printed and a
per-window `n_obs` record is kept so features can be weighted by evidence.

In [ ]:
# =============================================================================
# Cell 10 — Composite to windows, fill temporal gaps, save the cube
# =============================================================================
def assign_windows(times, windows):
    """Map observation times to window ids; -1 = outside every window."""
    t = pd.DatetimeIndex(times)
    out = np.full(len(t), -1, dtype=int)
    for _, w in windows.iterrows():
        out[(t >= w.win_start) & (t < w.win_next)] = w.win_id
    return out

def composite(values, win_idx, n_win, how):
    """Aggregate (n_obs, y, x) into (n_win, y, x). Empty windows stay NaN."""
    fn = {"median": np.nanmedian, "mean": np.nanmean}[how]
    out = np.full((n_win,) + values.shape[1:], np.nan, "float32")
    for w in range(n_win):
        sel = np.where(win_idx == w)[0]
        if len(sel):
            with np.errstate(all="ignore"):
                out[w] = fn(values[sel], axis=0)
    return out

def fill_time_gaps(cube, label):
    """Linear interpolation along time, per pixel; observed values untouched."""
    n = cube.shape[0]
    before = np.isfinite(cube).mean()
    filled = (pd.DataFrame(cube.reshape(n, -1))
                .interpolate(axis=0, limit_direction="both")
                .values.reshape(cube.shape).astype("float32"))
    observed = np.isfinite(cube)
    assert np.allclose(filled[observed], cube[observed])
    print(f"  {label:<5} valid {before:.1%} → {np.isfinite(filled).mean():.1%} "
          f"(interpolated {np.isfinite(filled).mean() - before:.1%})")
    return filled


for SITE in SITES:
    use_site(SITE)
    load_site_cfg()
    windows = pd.read_csv(P("windows.csv"), parse_dates=["win_start", "win_next",
                                                         "win_end", "win_center"])
    with xr.open_dataset(P("hls_scenes.nc")) as _s:
        scenes = _s.load()








    n_win = len(windows)
    print(f"compositing into {n_win} windows ({', '.join(f'{d} d' for d in sorted(windows.win_days.unique(), reverse=True))}):")
    idx = assign_windows(scenes.time.values, windows)
    cubes, observed = {}, {}
    for v in ("ndvi", "ndti"):
        cube = composite(scenes[v].values, idx, n_win, "median")
        observed[v] = np.isfinite(cube)                  # before interpolation
        print(f"  {v.upper():<5} {scenes.sizes['time']} obs days → "
              f"{int((~np.isnan(cube).all(axis=(1,2))).sum())}/{n_win} windows with data")
        cubes[v] = fill_time_gaps(cube, v.upper())

    # number of clear observation days per window, per pixel — the evidence behind
    # each composited value; 0 means the value is pure interpolation
    n_obs = np.zeros((n_win,) + scenes.ndvi.shape[1:], "int16")
    finite = np.isfinite(scenes.ndvi.values)
    for w in range(n_win):
        sel = np.where(idx == w)[0]
        if len(sel):
            n_obs[w] = finite[sel].sum(0)

    data_vars = {"ndvi": (("time", "y", "x"), cubes["ndvi"]),
                 "ndti": (("time", "y", "x"), cubes["ndti"]),
                 "n_obs": (("time", "y", "x"), n_obs)}
    grid30 = load_da(P("grid_30m.nc"))
    coords = {"time": pd.DatetimeIndex(windows.win_center),
              "y": grid30.y.values, "x": grid30.x.values}

    # --- ET, only when scenes exist --------------------------------------------
    eco_files = sorted(f for f in os.listdir(CACHE_ECO) if f.endswith(".nc"))
    if eco_files:
        eco_index = pd.read_csv(P("eco_index.csv"), parse_dates=["local"])
        gid_time = dict(zip(eco_index.gid, eco_index.local))
        vals, times = [], []
        for fn in eco_files:
            gid = fn[len(CFG["eco_var"]) + 1:-3]
            if gid not in gid_time:
                continue
            da = load_da(os.path.join(CACHE_ECO, fn))
            if np.isfinite(da.values).any():
                vals.append(da.values.astype("float32")); times.append(gid_time[gid])
        if vals:
            et_cube = composite(np.stack(vals), assign_windows(times, windows), n_win, "mean")
            print(f"  ET    {len(vals)} scenes → "
                  f"{int((~np.isnan(et_cube).all(axis=(1,2))).sum())}/{n_win} windows with data")
            et_cube = fill_time_gaps(et_cube, "ET")
            grid70 = load_da(P("grid_70m.nc"))
            data_vars["et"] = (("time", "y70", "x70"), et_cube)
            coords.update(y70=grid70.y.values, x70=grid70.x.values)
    else:
        print("  ET    no scenes — optical-only cube")

    fused = xr.Dataset(data_vars, coords=coords,
                       attrs={"site": SITE, "x_days": str(sorted(windows.win_days.unique())),
                              "crs": str(gpd.read_file(P("roi.geojson")).estimate_utm_crs()),
                              "created": pd.Timestamp.now().isoformat(timespec="seconds")})
    if os.path.exists(P("fused_cube.nc")):
        os.remove(P("fused_cube.nc"))
    fused.to_netcdf(P("fused_cube.nc"))
    print(f"\nfused cube: {n_win} time steps — " + ", ".join(
          f"{v} {tuple(fused[v].shape)}" for v in fused.data_vars))
    print(f"saved: fused_cube.nc  ({os.path.getsize(P('fused_cube.nc'))/1e6:.1f} MB)")

## Cell 11 — Ground-truth fields on the grid

The site's fields (from the store, not from CSB) are burned onto the 30 m grid
with the pixel-centre rule. Two rasters: `field_id` (row order in
`fields_site.gpkg`, 0 = outside) and `field_core`. Field pixels are
also shrunk by one pixel from the boundary for the per-field statistics —
CSB-derived boundaries are ±1 pixel and the mixed rim would otherwise blur the
contrast the labels are about.

In [ ]:
# =============================================================================
# Cell 11 — Burn the ground-truth fields onto the 30 m grid
# =============================================================================
from scipy import ndimage


for SITE in SITES:
    use_site(SITE)
    fields = gpd.read_file(P("fields_site.gpkg"))
    grid30 = load_da(P("grid_30m.nc")).rio.write_crs(
        gpd.read_file(P("roi.geojson")).estimate_utm_crs())
    fields_utm = fields.to_crs(grid30.rio.crs)
    fields_utm["field_id"] = np.arange(1, len(fields_utm) + 1, dtype="int32")

    shape = (grid30.sizes["y"], grid30.sizes["x"])
    fid = features.rasterize(zip(fields_utm.geometry, fields_utm.field_id),
                             out_shape=shape, transform=grid30.rio.transform(),
                             fill=0, dtype="int32", all_touched=False)

    # inner core: drop the one-pixel rim of every field
    core = np.zeros_like(fid)
    for i in fields_utm.field_id:
        m = fid == i
        core[ndimage.binary_erosion(m, structure=np.ones((3, 3)))] = i

    for name, arr in (("field_id", fid), ("field_core", core)):
        (xr.DataArray(arr.astype("int32"), coords=grid30.coords, dims=grid30.dims)
           .rio.write_crs(grid30.rio.crs).rio.write_nodata(None)
           .rio.to_raster(P(f"{name}.tif"), compress="DEFLATE"))
    fields_utm[["field_uid", "role", "field_id", "geometry"]].to_file(
        P("fields_site_utm.gpkg"), driver="GPKG")

    px = np.bincount(fid.ravel(), minlength=len(fields_utm) + 1)[1:]
    pxc = np.bincount(core.ravel(), minlength=len(fields_utm) + 1)[1:]
    print(f"{len(fields_utm)} fields burned onto {shape[0]} x {shape[1]} px")
    for (_, f), n, nc in zip(fields_utm.iterrows(), px, pxc):
        print(f"  {f.field_uid:<26} {f.role:<20} {n:>5} px  core {nc:>5} px  "
              f"({n * 0.09:.1f} ha)")
    assert (pxc > 0).all(), "a field has no core pixels — too small for 30 m analysis"
    print("saved: field_id.tif, field_core.tif, fields_site_utm.gpkg")

## Cell 11b — Detection windows; the interseeded season gets two

A fall-seeded season keeps one window, spring after seeding → termination.
The interseeded season (2022-23) is split at the **corn harvest**, taken from
the control field's HLS NDVI (the cover-crop field can stay green after
harvest): the last clear scene at or above `harvest_ndvi` (0.40) closes the
**corn window** (interseeding → canopy closure → senescence; GCVI is the index
to watch, since chlorophyll under a thinning canopy is the only way the cover
crop can show), and the first clear scene below it opens the **post-harvest
window** (→ termination). Scenes in between are the harvest itself and belong
to neither. The dates can be fixed by hand in `CFG["windows"]["harvest_override"]`.
The v8 layers (Cells 13-17) use the post-harvest window for this season.

In [ ]:
# =============================================================================
# Cell 11b — Detection windows per season; interseeded seasons get two
# =============================================================================
# Fall-seeded seasons keep one window: spring after seeding → termination.
# An interseeded season is split at the corn harvest, found in the HLS record
# of the control field(s) (the cover-crop field may stay green after harvest):
#   corn        interseeding → last clear scene with NDVI ≥ harvest_ndvi
#               (cover crop under a closing, then senescing corn canopy)
#   postharvest first clear scene with NDVI < harvest_ndvi → termination
#               (cover crop exposed after harvest)
# The scenes in between (the harvest itself) belong to neither window.
# season_windows.csv then carries the post-harvest window for the interseeded
# season, so the v8 layers (Cells 13-17) describe the exposed cover crop.


def field_series(stack_fn, t, core, fid_ids, min_clear):
    """Field means of an index over core pixels, per scene; NaN when < min_clear clear."""
    out = {}
    for i in fid_ids:
        m = core == i
        vals = []
        for k in range(len(t)):
            v = stack_fn(k)[m]
            ok = np.isfinite(v)
            vals.append(float(v[ok].mean()) if ok.mean() >= min_clear else np.nan)
        out[i] = np.array(vals)
    return out


for SITE in SITES:
    use_site(SITE)
    R = CFG["windows"]
    sw = pd.read_csv(P("season_windows.csv"))
    fields = gpd.read_file(P("fields_site_utm.gpkg"))
    core = read_tif(P("field_core.tif")).astype("int32")
    with xr.open_dataset(P("hls_bands.nc")) as _b:
        t = pd.DatetimeIndex(_b.time.values)
        red, nir = _b.red.values, _b.nir.values
    with np.errstate(all="ignore"):
        ndvi = lambda k: (nir[k] - red[k]) / (nir[k] + red[k])
    ctrl = fields[fields.role.str.contains("control")].field_id.tolist() or fields.field_id.tolist()

    rows = []
    for i, w in sw.iterrows():
        season = w.season
        y0 = int(season[:4]); y1 = y0 + 1
        ts = lambda s: pd.Timestamp(s.replace("y0", str(y0)).replace("y1", str(y1)))
        start, end = pd.Timestamp(w.window_start), pd.Timestamp(w.window_end)
        if "interseed" not in str(w.seeding_method).lower():
            rows.append(dict(season=season, window="spring", start=start.date(), end=end.date(),
                             source=w.start_source))
            continue
        # --- corn harvest from the control field(s) --------------------------
        h0, h1 = ts(R["harvest_search"][0]), ts(R["harvest_search"][1])
        sel = np.where((t >= h0) & (t <= h1))[0]
        tt = t[sel]
        series = field_series(lambda k: ndvi(sel[k]), tt, core, ctrl, CFG["feat"]["min_clear"])
        mean_ctrl = np.nanmean(np.vstack(list(series.values())), axis=0) if series else np.array([])
        override = (R.get("harvest_override") or {}).get(season)
        if override:
            last_up, first_down = pd.Timestamp(override[0]), pd.Timestamp(override[1])
            src = "harvest set in CFG"
        else:
            obs = np.isfinite(mean_ctrl)
            peak = int(np.nanargmax(mean_ctrl)) if obs.any() else 0
            below = [k for k in range(peak, len(tt)) if obs[k] and mean_ctrl[k] < R["harvest_ndvi"]]
            if not below:
                raise RuntimeError(f"{season}: no control-field scene below NDVI {R['harvest_ndvi']} "
                                   f"between {h0.date()} and {h1.date()} — set CFG['windows']"
                                   f"['harvest_override']['{season}'] = (last_corn_date, first_bare_date)")
            kd = below[0]
            ku = max(k for k in range(peak, kd) if obs[k]) if any(obs[peak:kd]) else peak
            last_up, first_down = tt[ku], tt[kd]
            src = (f"control NDVI {mean_ctrl[ku]:.2f} on {last_up:%Y-%m-%d} → "
                   f"{mean_ctrl[kd]:.2f} on {first_down:%Y-%m-%d} (threshold {R['harvest_ndvi']})")
        rows.append(dict(season=season, window="corn", start=start.date(), end=last_up.date(),
                         source=f"interseeding → last corn scene; {src}"))
        rows.append(dict(season=season, window="postharvest", start=first_down.date(), end=end.date(),
                         source=f"first post-harvest scene → termination; {src}"))
        sw.loc[i, "interseed_date"] = str(start.date())
        sw.loc[i, "harvest_between"] = f"{last_up.date()} … {first_down.date()}"
        sw.loc[i, "window_start"] = str(first_down.date())
        sw.loc[i, "days"] = (end - first_down).days + 1
        sw.loc[i, "start_source"] = "corn harvest (post-harvest window)"
        print(f"{season}: control-field NDVI {h0:%m-%d} … {h1:%m-%d}: " +
              ", ".join(f"{d:%m-%d} {v:.2f}" for d, v in zip(tt, mean_ctrl) if np.isfinite(v)))

    dw = pd.DataFrame(rows)
    dw["days"] = [(pd.Timestamp(e) - pd.Timestamp(s)).days + 1 for s, e in zip(dw.start, dw.end)]
    dw.to_csv(P("detect_windows.csv"), index=False)
    sw.to_csv(P("season_windows.csv"), index=False)
    print("\ndetection windows (the same dates for every field of a season):")
    for _, r in dw.iterrows():
        print(f"  {r.season}  {r.window:<12} {r.start} → {r.end}  ({r.days:>3} d)  {r.source}")
    print("saved: detect_windows.csv; season_windows.csv (interseeded season → post-harvest window)")


## Cell 12 — CSB fields and cropland per season

USDA Crop Sequence Boundaries (2018–2025) give field polygons with the CDL
crop of every year. They serve two purposes: the **cropland mask** of each
season (the field grew corn or soybean in either year of the season — the
v5_3 rule) and the **units for field averaging** in Cell 15. The ROI's CSB
polygons are cut once from the national geodatabase (downloaded from USDA
NASS, ~2 min) and cached as
`csb_roi.gpkg`, so later runs never touch the 3.8 GB file. The ground-truth
fields are cropland by definition.

In [ ]:
# =============================================================================
# Cell 12 — CSB field boundaries and a per-season cropland mask
# =============================================================================
import glob, pyogrio


def ensure_csb():
    """Path of the national CSB geodatabase on local disk, fetching it once.

    First choice: USDA NASS's own download (3.8 GB, ~35 MB/s from CyVerse,
    about 2 min). Fallback: the copy on the Data Store, read in 16 MB blocks
    (the FUSE mount has run anywhere from 0.3 to 30 MB/s). Either way the zip
    is extracted and deleted. Sites that already have csb_roi.gpkg never call
    this.
    """
    found = glob.glob(os.path.join(CFG["csb_dir"], "**", "*.gdb"), recursive=True)
    if found:
        return found[0]
    os.makedirs(CFG["csb_dir"], exist_ok=True)
    local = os.path.join(CFG["csb_dir"], "csb.zip")
    t0 = time.time()
    try:
        with requests.get(CFG["csb_url"], stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(local, "wb") as fo:
                for ch in r.iter_content(16 * 1024 * 1024):
                    fo.write(ch)
        src = "USDA NASS"
    except Exception as e:
        print(f"  NASS download failed ({type(e).__name__}) — copying from the Data Store")
        with open(CFG["csb_zip"], "rb") as fi, open(local, "wb") as fo:
            while True:
                b = fi.read(16 * 1024 * 1024)
                if not b:
                    break
                fo.write(b)
        src = "Data Store"
    print(f"  CSB zip from {src}: {os.path.getsize(local)/1e9:.1f} GB in {(time.time()-t0)/60:.1f} min")
    with zipfile.ZipFile(local) as zf:
        zf.extractall(CFG["csb_dir"])
    os.remove(local)
    return glob.glob(os.path.join(CFG["csb_dir"], "**", "*.gdb"), recursive=True)[0]


for SITE in SITES:
    use_site(SITE)
    roi = gpd.read_file(P("roi.geojson"))
    utm = roi.estimate_utm_crs()
    grid30 = load_da(P("grid_30m.nc")).rio.write_crs(utm)
    labels = pd.read_csv(P("seasons_site.csv"))
    fid = rxr.open_rasterio(P("field_id.tif")).squeeze(drop=True).values
    shape = (grid30.sizes["y"], grid30.sizes["x"])

    # --- CSB polygons inside the ROI (cached per site, archived) -------------
    if not have("csb_roi.gpkg"):
        gdb = ensure_csb()
        crs = pyogrio.read_info(gdb, layer=CFG["csb_layer"])["crs"]
        roi_csb = roi.to_crs(crs)
        csb = gpd.read_file(gdb, layer=CFG["csb_layer"], engine="pyogrio",
                            bbox=tuple(roi_csb.total_bounds))
        keep = ["CSBID", "CSBACRES"] + [c for c in csb.columns if re.fullmatch(r"CDL\d{4}", c)]
        csb = gpd.clip(csb[keep + ["geometry"]], roi_csb.geometry.iloc[0])
        csb.to_crs(utm).to_file(P("csb_roi.gpkg"), driver="GPKG")
    csb = gpd.read_file(P("csb_roi.gpkg")).to_crs(utm).reset_index(drop=True)
    csb["csb_px"] = np.arange(1, len(csb) + 1, dtype="int32")
    cdl_cols = sorted(c for c in csb.columns if re.fullmatch(r"CDL\d{4}", c))
    for c in cdl_cols:
        csb[c] = pd.to_numeric(csb[c], errors="coerce")

    csb_id = features.rasterize(zip(csb.geometry, csb.csb_px), out_shape=shape,
                                transform=grid30.rio.transform(), fill=0, dtype="int32")
    write_tif(P("csb_id.tif"), csb_id, grid30)

    # --- cropland per season: the field grew corn or soy in either year ------
    # of the season (fall year y0, spring year y1). The ground-truth fields are
    # cropland by definition, whatever CSB says about them.
    print(f"{len(csb)} CSB fields in the ROI · CDL years {cdl_cols[0][3:]}–{cdl_cols[-1][3:]}")
    for season in sorted(labels.season.unique()):
        y0 = int(season[:4]); cols = [f"CDL{y}" for y in (y0, y0 + 1) if f"CDL{y}" in csb]
        if not cols:
            print(f"  {season}: no CSB year covers this season — cropland = ground-truth fields only")
            crop_ids = []
        else:
            crop_ids = csb.loc[csb[cols].isin(CFG["crop_codes"]).any(axis=1), "csb_px"].values
        mask = np.isin(csb_id, crop_ids) | (fid > 0)
        write_tif(P(f"cropland_{season}.tif"), mask.astype("int32"), grid30)
        print(f"  {season}: cropland {mask.mean():.0%} of the ROI "
              f"({len(crop_ids)} corn/soy fields, CDL {', '.join(c[3:] for c in cols)})")
    print("saved: csb_roi.gpkg, csb_id.tif, cropland_<season>.tif")


## Cell 13 — Look at the cube (v5_2 inspection)

For the most recent labelled season: NDVI at the summer peak and mid-winter,
winter NDTI, spring ET, the cropland mask, the winter NDVI × NDTI plane, and
ROI-wide trajectories with every season's off-season and termination windows
shaded. Printed checks: summer cropland NDVI > 0.7, winter NDTI in 0.02–0.20.

In [ ]:
# =============================================================================
# Cell 13 — Visual inspection of the fused cube (NDVI · NDTI · ET), latest season
# =============================================================================
# The v5_2 check, per site, for the most recent labelled season: channel maps,
# the cropland mask, the winter NDVI × NDTI plane and ROI-wide trajectories.
for SITE in SITES:
    use_site(SITE)
    with xr.open_dataset(P("fused_cube.nc")) as _f:
        fused = _f.load()
    labels = pd.read_csv(P("seasons_site.csv"))
    season = sorted(labels.season.unique())[-1]
    D = season_window(season)
    cropland = read_tif(P(f"cropland_{season}.tif")).astype(bool)
    t = pd.DatetimeIndex(fused.time.values)
    near = lambda d: int(np.argmin(np.abs(t - pd.Timestamp(d))))
    i_summer = near(f"{D['y1']}-07-15") if t[-1] >= pd.Timestamp(f"{D['y1']}-07-01") else near(f"{D['y0']}-07-15")
    i_winter, i_spring = near(f"{D['y1']}-01-15"), near(D["end"] - pd.Timedelta(days=21))
    ndvi, ndti = fused.ndvi.values, fused.ndti.values

    fig, ax = plt.subplots(3, 3, figsize=(16, 13))
    for a, img, ttl, cm, vr in [
            (ax[0, 0], ndvi[i_summer], f"NDVI — summer peak\n{t[i_summer]:%Y-%m-%d}", "RdYlGn", (0, 0.9)),
            (ax[0, 1], ndvi[i_winter], f"NDVI — mid-winter\n{t[i_winter]:%Y-%m-%d}", "RdYlGn", (0, 0.6)),
            (ax[0, 2], ndti[i_winter], f"NDTI — mid-winter\n{t[i_winter]:%Y-%m-%d}", "copper_r", (0.0, 0.25))]:
        im = a.imshow(img, cmap=cm, vmin=vr[0], vmax=vr[1])
        a.set_title(ttl, fontsize=10); plt.colorbar(im, ax=a, fraction=0.046)

    if "et" in fused:
        et_img = fused.et.values[i_spring]
        im = ax[1, 0].imshow(et_img, cmap="YlGnBu")
        ax[1, 0].set_title(f"ET — spring\n{t[i_spring]:%Y-%m-%d} · valid {np.isfinite(et_img).mean():.0%}", fontsize=10)
        plt.colorbar(im, ax=ax[1, 0], fraction=0.046)
    else:
        ax[1, 0].text(0.5, 0.5, "no ECOSTRESS for this period", ha="center", va="center")
    ax[1, 1].imshow(cropland, cmap="Greens", interpolation="nearest")
    ax[1, 1].set_title(f"CSB cropland {season} — {cropland.mean():.0%} of ROI", fontsize=10)
    ax[0, 0].figure.text(0.5, 1.005, f"detection window {season}: {D['label']}", ha="center", fontsize=10)

    a = ax[1, 2]
    sel = cropland & np.isfinite(ndvi[i_winter]) & np.isfinite(ndti[i_winter])
    xs, ys = ndvi[i_winter][sel], ndti[i_winter][sel]
    if xs.size > 20000:
        k = np.random.default_rng(0).choice(xs.size, 20000, replace=False); xs, ys = xs[k], ys[k]
    a.hexbin(xs, ys, gridsize=45, cmap="magma_r", mincnt=1, bins="log")
    a.axvline(CFG["green"], color="#2b7a3d", lw=1, ls="--")
    a.set_xlabel("NDVI"); a.set_ylabel("NDTI")
    a.set_title("winter cropland: NDVI × NDTI\nright = green · top = residue", fontsize=10)
    a.grid(alpha=0.25)

    with np.errstate(all="ignore"):
        crop_ndvi  = np.nanmean(np.where(cropland, ndvi, np.nan), axis=(1, 2))
        other_ndvi = np.nanmean(np.where(~cropland, ndvi, np.nan), axis=(1, 2))
        crop_ndti  = np.nanmean(np.where(cropland, ndti, np.nan), axis=(1, 2))
        other_ndti = np.nanmean(np.where(~cropland, ndti, np.nan), axis=(1, 2))
        et_ts = np.nanmean(fused.et.values, axis=(1, 2)) if "et" in fused else np.full(len(t), np.nan)
    for a, series, ylab in [
            (ax[2, 0], [(crop_ndvi, "#2b7a3d", "cropland"), (other_ndvi, "#8a99a5", "other")], "NDVI"),
            (ax[2, 1], [(crop_ndti, "#8a5a2b", "cropland"), (other_ndti, "#8a99a5", "other")], "NDTI"),
            (ax[2, 2], [(et_ts, "#c4763a", "ROI mean ET")], "ET")]:
        for y, c, lab in series:
            a.plot(t, y, "-", lw=1, color=c, label=lab)
        for s in sorted(labels.season.unique()):
            Ds = season_window(s)
            a.axvspan(*Ds["win"], color="#4a7ec4", alpha=0.10)
            a.axvline(Ds["term"], color="#c44", lw=0.8, ls="--")
        a.xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 4, 7, 10)))
        a.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        a.xaxis.set_minor_locator(mdates.MonthLocator())
        a.grid(alpha=0.35, which="major"); a.grid(alpha=0.12, which="minor")
        a.set_ylabel(ylab); a.set_title(f"{ylab} trajectory (blue = detection window, red = termination)", fontsize=10)
        a.legend(fontsize=8); a.tick_params(axis="x", rotation=60, labelsize=7)
    for a in (ax[0, 0], ax[0, 1], ax[0, 2], ax[1, 0], ax[1, 1]):
        a.set_xticks([]); a.set_yticks([])
    plt.suptitle(f"{SITE} — fused cube QC, season {season}", y=1.0)
    plt.tight_layout()
    fig_path = os.path.join(FIG_DIR, f"fused_cube_qc_{SITE}.png")
    plt.savefig(fig_path, dpi=150, bbox_inches="tight"); plt.show()
    print(f"saved: {fig_path}")
    print(f"summer cropland NDVI {crop_ndvi[i_summer]:.3f} (expect > 0.7) · "
          f"winter cropland NDVI {crop_ndvi[i_winter]:.3f} vs other {other_ndvi[i_winter]:.3f} · "
          f"winter cropland NDTI {crop_ndti[i_winter]:.3f}")


## Cell 14 — The labelled fields: maps and per-season trajectories

Maps of the last season three weeks before termination. Then one panel per
season, June of the fall year to July of the spring year, with monthly ticks:
field-mean NDVI of every labelled field (dots = composites that were really
observed, lines include interpolation), the detection window shaded, the
seeding and termination dates marked, and underneath the cover-crop field
minus the control. This is where a label that contradicts the imagery shows
up — read it before trusting any number from Cell 17.

In [ ]:
# =============================================================================
# Cell 14 — The labelled fields: maps and per-season trajectories
# =============================================================================
ROLE_COLOR = {"cc_treatment": "#2b7a3d", "candidate_same_farm": "#7fbf7b",
              "nt_control": "#8a99a5", "neighbour_control": "#8a99a5",
              "trial_host_field": "#c4763a"}

for SITE in SITES:
    use_site(SITE)
    with xr.open_dataset(P("fused_cube.nc")) as _f:
        fused = _f.load()
    fields = gpd.read_file(P("fields_site_utm.gpkg"))
    labels = pd.read_csv(P("seasons_site.csv"))
    sw = pd.read_csv(P("season_windows.csv"))
    fid  = rxr.open_rasterio(P("field_id.tif")).squeeze(drop=True).values
    core = rxr.open_rasterio(P("field_core.tif")).squeeze(drop=True).values
    t = pd.DatetimeIndex(fused.time.values)
    ndvi, ndti = fused.ndvi.values, fused.ndti.values

    # field-mean NDVI on core pixels, and which composites were really observed
    # (≥ half of the core pixels had a clear scene in that window)
    traj, seen = {}, {}
    for _, f in fields.iterrows():
        m = core == f.field_id
        traj[f.field_uid] = np.nanmean(ndvi[:, m], axis=1)
        seen[f.field_uid] = (fused.n_obs.values[:, m] > 0).mean(axis=1) >= 0.5

    # --- 1. maps: last season, 3 weeks before termination ------------------
    D = season_window(sorted(sw.season)[-1])
    i_sp = int(np.argmin(np.abs(t - (D["end"] - pd.Timedelta(days=21)))))
    edge = np.zeros_like(fid, bool)
    edge[:, 1:] |= fid[:, 1:] != fid[:, :-1]; edge[1:, :] |= fid[1:, :] != fid[:-1, :]
    edge_img = np.ma.masked_where(~edge, np.ones_like(fid, "float32"))
    i_su = int(np.argmin(np.abs(t - pd.Timestamp(f"{D['y0']}-07-20"))))
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for a, img, ttl, cm, vr in [
            (axes[0], ndvi[i_sp], f"NDVI {t[i_sp]:%Y-%m-%d} (3 wk before termination)", "RdYlGn", (0, 0.8)),
            (axes[1], ndvi[i_su], f"NDVI {t[i_su]:%Y-%m-%d} (summer)", "RdYlGn", (0, 0.9)),
            (axes[2], ndti[i_sp], f"NDTI {t[i_sp]:%Y-%m-%d}", "copper_r", (0, 0.25))]:
        im = a.imshow(img, cmap=cm, vmin=vr[0], vmax=vr[1])
        a.imshow(edge_img, cmap="gray_r", vmin=0, vmax=1, alpha=0.6, interpolation="nearest")
        for _, f in fields.iterrows():
            yy, xx = np.where(fid == f.field_id)
            if len(yy):
                a.text(xx.mean(), yy.mean(), f.field_uid.split("_")[-1], fontsize=8, ha="center",
                       va="center", color=ROLE_COLOR.get(f.role, "k"),
                       bbox=dict(fc="white", ec="none", alpha=0.6, pad=0.5))
        a.set_title(ttl, fontsize=10); a.set_xticks([]); a.set_yticks([])
        plt.colorbar(im, ax=a, fraction=0.046)
    plt.suptitle(f"{SITE} — season {D['season']}, detection window {D['label']}")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"qc_{SITE}.png"), dpi=150, bbox_inches="tight"); plt.show()

    # --- 2. trajectories: one panel per season, Jun y0 → Jul y1 --------------
    # top: field-mean NDVI (dots = observed composite, line = incl. interpolation)
    # bottom: CC field minus control field
    seasons = sorted(sw.season)
    ncol = 3; nrow = int(np.ceil(len(seasons) / ncol))
    fig = plt.figure(figsize=(6.2 * ncol, 6.0 * nrow))
    # rows per season block: NDVI, difference, spacer (room for the next title)
    gs = fig.add_gridspec(3 * nrow, ncol, height_ratios=[2.2, 1, 0.55] * nrow, hspace=0.12)
    cc_uid = [u for u, r in zip(fields.field_uid, fields.role) if "cc" in r or "candidate" in r]
    ct_uid = [u for u, r in zip(fields.field_uid, fields.role) if "control" in r]
    for k, s in enumerate(seasons):
        D = season_window(s)
        x0, x1 = pd.Timestamp(f"{D['y0']}-06-01"), pd.Timestamp(f"{D['y1']}-07-31")
        sel = (t >= x0) & (t <= x1)
        a = fig.add_subplot(gs[3 * (k // ncol), k % ncol])
        b = fig.add_subplot(gs[3 * (k // ncol) + 1, k % ncol], sharex=a)
        lab = labels[labels.season == s].set_index("field_uid").cc_present
        for _, f in fields.iterrows():
            c = ROLE_COLOR.get(f.role, "k")
            a.plot(t[sel], traj[f.field_uid][sel], "-", color=c, lw=1.4,
                   label=f"{f.field_uid.split('_')[-1]} (CC={lab.get(f.field_uid, '?')})")
            o = sel & seen[f.field_uid]
            a.plot(t[o], traj[f.field_uid][o], "o", ms=3.5, color=c)
        if cc_uid and ct_uid:
            diff = np.nanmean([traj[u] for u in cc_uid], 0) - np.nanmean([traj[u] for u in ct_uid], 0)
            b.axhline(0, color="k", lw=0.6)
            b.fill_between(t[sel], 0, diff[sel], where=diff[sel] > 0, color="#2b7a3d", alpha=0.35, lw=0)
            b.fill_between(t[sel], 0, diff[sel], where=diff[sel] <= 0, color="#8a99a5", alpha=0.35, lw=0)
            b.plot(t[sel], diff[sel], "-", color="#2b7a3d", lw=1)
            lim = max(0.15, float(np.nanmax(np.abs(diff[sel]))) * 1.1)
            b.set_ylabel("CC − control", fontsize=8); b.set_ylim(-lim, lim)
        w = sw.set_index("season").loc[s]
        for ax_ in (a, b):
            ax_.axvspan(D["start"], D["end"], color="#4a7ec4", alpha=0.12)
            if pd.notna(D["seed"]):
                ax_.axvline(D["seed"], color="#2b7a3d", lw=1, ls="--")
            ax_.axvline(D["term"], color="#c44", lw=1, ls="--")
            ax_.xaxis.set_major_locator(mdates.MonthLocator())
            ax_.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
            ax_.xaxis.set_minor_locator(mdates.WeekdayLocator(byweekday=0))
            ax_.grid(alpha=0.35, which="major"); ax_.grid(alpha=0.10, which="minor")
            ax_.set_xlim(x0, x1)
        plt.setp(a.get_xticklabels(), visible=False)
        b.tick_params(axis="x", labelsize=7, rotation=90)
        a.set_ylim(0.1, 0.95); a.set_ylabel("field-mean NDVI", fontsize=8)
        a.set_title(f"{s} · window {D['label']} ({w.days} d)\n{w.seeding_method[:45]}",
                    fontsize=9, loc="left")
        if k == 0:
            a.legend(fontsize=7, loc="upper right")
    fig.suptitle(f"{SITE}: field-mean NDVI per season — blue = detection window, green dashes = CC "
                 f"seeding, red dashes = termination, dots = observed composites (lines include "
                 f"interpolation)", fontsize=10, y=0.995)
    fig_path = os.path.join(FIG_DIR, f"trajectories_{SITE}.png")
    plt.savefig(fig_path, dpi=150, bbox_inches="tight"); plt.show()
    print(f"saved: figures/qc_{SITE}.png, {fig_path}")

    # quick numbers behind the picture: mean NDVI inside each season's window
    print(f"\n{'season':<9}{'window':<26}" + "".join(f"{u.split('_')[-1]:>9}" for u in fields.field_uid)
          + f"{'observed':>10}")
    for s in seasons:
        D = season_window(s); inwin = (t >= D["start"]) & (t <= D["end"])
        obs = np.mean([seen[u][inwin].mean() for u in fields.field_uid])
        print(f"{s:<9}{D['label']:<26}" + "".join(f"{np.nanmean(traj[u][inwin]):>9.3f}"
              for u in fields.field_uid) + f"{obs:>10.0%}")


## Cell 15 — Time series → firerx layers, one set per season

firerx has **no time dimension**: `create_pyramid_set.py` stacks layers. So
each season's stretch of the cube is compressed into scalars per pixel, one
GeoTIFF per scalar — the v5_2 layers, now read inside **that season's
detection window**, which Cell 3 derives from the paper's cover-crop dates
(`season_windows.csv`, printed by Cell 3 and Cell 15, and stored in every
GeoTIFF's tags):

| season type | window | example (IN) |
|---|---|---|
| fall-seeded (drilled after harvest) | spring after seeding: Mar 1 → termination | rye 2021-11-08 → window 2022-03-01 … 2022-05-16 |
| interseeded into the cash crop | seeding → termination | mix 2022-06-27 → window 2022-06-27 … 2023-05-18 |
| dates not reported | Mar 1 → May 15 | 2018-19 … 2020-21 |

The same window is used for every field of the season, so the control is
read over exactly the cover-crop field's dates.

| layer | what it measures | read over |
|---|---|---|
| `n_green` | days with NDVI > `green` | detection window |
| `ndvi_int` | NDVI integrated over days / 100 | detection window |
| `peak_win` | days from the window start to the NDVI peak | detection window |
| `up_slope` | steepest window-to-window NDVI rise | last 75 d before termination |
| `term_drop` | steepest NDVI fall | termination − 24 d … + 40 d |
| `spring_rel` | NDVI minus the season's cropland median | last 6 weeks before termination |
| `et_spring` | mean ET, 70 m → 30 m (only with ECOSTRESS) | last 6 weeks before termination |

`n_green` counts days, not composites, so 8- and 16-day composites compare.
Every layer is NaN-free and zero outside cropland (firerx's
`roundup_layer_1a` rejects a sample with any NaN). Two versions:
`layers/<season>/pixel/` (raw) and `layers/<season>/` (mean within CSB fields
≥ `min_field_px`; the firerx input, as in v5_2). `fields_features.gpkg`
holds the field means as a table, one layer per season.

In [ ]:
# =============================================================================
# Cell 15 — Compress each season's time axis into firerx layers
# =============================================================================
from scipy import ndimage

def season_layers(ndvi, t, win_days, D, cropland, green):
    """The phenology layers for ONE season, per pixel, NaN-free.

    Every layer is read inside that season's detection window D (from the
    paper's cover-crop dates, see season_windows.csv):
      n_green, ndvi_int, peak_win  → D["win"]   (start … termination)
      up_slope                     → D["rise"]  (last 75 d before termination)
      term_drop                    → D["termw"] (termination − 24 d … + 40 d)
      spring_rel                   → D["late"]  (last 6 weeks before termination)
    ndvi      (n_win, y, x), gaps already filled by Cell 10
    win_days  (n_win,) length of each composite window (8 or 16 days)
    """
    ndvi = np.nan_to_num(ndvi, nan=0.0)
    sel = lambda a, b: (t >= a) & (t <= b)
    win, rise, termw, late = sel(*D["win"]), sel(*D["rise"]), sel(*D["termw"]), sel(*D["late"])
    for nm, s in (("detection", win), ("rise", rise), ("termination", termw), ("late", late)):
        if s.sum() < 2:
            return None, f"{nm} window covers only {s.sum()} time steps"
    w = win_days[:, None, None].astype("float32")
    f = {}
    # 1. days with NDVI > green inside the detection window (days, not
    #    windows, so 8- and 16-day composites compare)
    f["n_green"] = ((ndvi[win] > green) * w[win]).sum(0)
    # 2. NDVI integrated over the window's days / 100
    f["ndvi_int"] = (ndvi[win] * w[win]).sum(0) / 100.0
    # 3. steepest rise before termination, steepest fall around it
    d = np.diff(ndvi, axis=0)
    f["up_slope"]  = d[rise[1:]].max(0)
    f["term_drop"] = -d[termw[1:]].min(0)
    # 4. days from the window start to the NDVI peak inside the window
    idx = np.argmax(np.where(win[:, None, None], ndvi, -9.0), axis=0)
    f["peak_win"] = (t[idx.ravel()] - D["start"]).days.values.reshape(idx.shape)
    # 5. NDVI of the last weeks before termination relative to the season's
    #    cropland median (cancels the season's weather)
    spr = ndvi[late].mean(0)
    f["spring_rel"] = spr - np.median(spr[cropland]) if cropland.any() else spr
    return {k: np.asarray(v, "float32") for k, v in f.items()}, None


def field_mean(arr, csb_id, ids):
    """Average `arr` inside each CSB field of `ids`; pixels outside keep their value."""
    out = arr.copy()
    lut = np.zeros(int(csb_id.max()) + 1, "float32")
    lut[ids] = ndimage.mean(arr, csb_id, ids)
    inside = np.isin(csb_id, ids)
    out[inside] = lut[csb_id[inside]]
    return out


for SITE in SITES:
    use_site(SITE)
    with xr.open_dataset(P("fused_cube.nc")) as _f:
        fused = _f.load()
    roi = gpd.read_file(P("roi.geojson")); utm = roi.estimate_utm_crs()
    grid30 = load_da(P("grid_30m.nc")).rio.write_crs(utm)
    t = pd.DatetimeIndex(fused.time.values)
    win_days = pd.read_csv(P("windows.csv")).win_days.values
    labels = pd.read_csv(P("seasons_site.csv"))
    csb = gpd.read_file(P("csb_roi.gpkg")).to_crs(utm).reset_index(drop=True)
    csb["csb_px"] = np.arange(1, len(csb) + 1, dtype="int32")
    csb_id = read_tif(P("csb_id.tif")).astype("int32")
    has_et = "et" in fused
    if has_et:                      # ET onto the 30 m grid once, for the ET layer
        et30 = (xr.DataArray(fused.et.values, dims=("time", "y", "x"),
                             coords={"time": t, "y": fused.y70.values, "x": fused.x70.values})
                  .rio.write_crs(utm).rio.set_spatial_dims("x", "y")
                  .rio.reproject_match(grid30, resampling=Resampling.bilinear).values)

    ff_path = P("fields_features.gpkg")
    if os.path.exists(ff_path):
        os.remove(ff_path)
    print(f"{'season':<9}{'layer':<11}{'cropland mean':>14}{'sd':>8}{'range':>18}")
    for season in sorted(labels.season.unique()):
        D = season_window(season)
        cropland = read_tif(P(f"cropland_{season}.tif")).astype(bool)
        feat, why = season_layers(fused.ndvi.values, t, win_days, D, cropland, CFG["green"])
        if feat is None:
            print(f"{season:<9}skipped: {why}")
            continue
        names = list(LAYERS)
        if has_et:
            late = (t >= D["late"][0]) & (t <= D["late"][1])
            feat["et_spring"] = np.nanmean(et30[late], axis=0).astype("float32")
            names.append("et_spring")
        inwin = (t >= D["start"]) & (t <= D["end"])
        obs = (fused.n_obs.values[inwin][:, cropland] > 0).mean()
        print(f"{season:<9}window {D['label']} · {int(inwin.sum())} composites · "
              f"{obs:.0%} of cropland pixel-composites observed (rest interpolated)")
        tags = dict(season=season, window_start=str(D["start"].date()), window_end=str(D["end"].date()),
                    green=str(CFG["green"]))

        # field means over CSB fields big enough to be a field (>= min_field_px)
        px = np.bincount(csb_id.ravel(), minlength=len(csb) + 1)
        ids = np.array([i for i in csb.csb_px if px[i] >= CFG["min_field_px"]
                        and cropland[csb_id == i].mean() > 0.5], "int32")
        out_f = P(os.path.join("layers", season))
        out_p = os.path.join(out_f, "pixel")
        os.makedirs(out_p, exist_ok=True)
        for name in names:
            a_px = np.nan_to_num(feat[name], nan=0.0, posinf=0.0, neginf=0.0)
            a_fd = field_mean(a_px, csb_id, ids) if len(ids) else a_px
            a_px[~cropland] = 0.0; a_fd[~cropland] = 0.0
            write_tif(os.path.join(out_p, f"{name}.tif"), a_px, grid30, tags)   # per pixel
            write_tif(os.path.join(out_f, f"{name}.tif"), a_fd, grid30, tags)   # per field → firerx
            v = a_fd[cropland]
            print(f"{season:<9}{name:<11}{v.mean():>14.3f}{v.std():>8.3f}"
                  f"{v.min():>9.2f} – {v.max():<7.2f}")
        write_tif(os.path.join(out_f, "cropland.tif"), cropland.astype("float32"), grid30)

        # the same numbers as a table: one row per CSB field
        rows = csb.loc[csb.csb_px.isin(ids), ["CSBID", "CSBACRES", "csb_px", "geometry"]].copy()
        for name in names:
            rows[name] = ndimage.mean(np.nan_to_num(feat[name]), csb_id, rows.csb_px.values)
        rows["window_start"], rows["window_end"] = str(D["start"].date()), str(D["end"].date())
        rows.to_file(ff_path, layer=f"s{season.replace('-', '_')}", driver="GPKG")
        print(f"{season:<9}{len(ids)} CSB fields averaged · layers/{season}/ (field) + /pixel/")

print(f"\nGREEN = {CFG['green']} drives n_green; windows in season_windows.csv (also stored as GeoTIFF tags).")
print("firerx: point batch_align_raster at layers/<season>/ (NaN-free, one GeoTIFF per layer)")


## Cell 16 — Are the layers each earning their place? (v5_2 diagnostics)

Per season: the most redundant pair of layers (firerx sample survival decays
with every extra layer, so a duplicate is pure cost), `n_green` per pixel and
per field with the ground-truth fields outlined (real signal is field-shaped,
speckle is noise), and the share of cropland whose off-season NDVI maximum
exceeds each threshold (a step marks two populations; a smooth slide means
the threshold alone does not split them). The mean correlation matrix across
seasons is printed.

In [ ]:
# =============================================================================
# Cell 16 — Layer diagnostics: redundancy, spatial coherence, threshold
# =============================================================================
for SITE in SITES:
    use_site(SITE)
    labels = pd.read_csv(P("seasons_site.csv"))
    fid = rxr.open_rasterio(P("field_id.tif")).squeeze(drop=True).values
    with xr.open_dataset(P("fused_cube.nc")) as _f:
        fused = _f.load()
    t = pd.DatetimeIndex(fused.time.values)
    seasons = [s for s in sorted(labels.season.unique())
               if os.path.isfile(P(f"layers/{s}/pixel/n_green.tif"))]
    edge = np.zeros_like(fid, bool)
    edge[:, 1:] |= fid[:, 1:] != fid[:, :-1]; edge[1:, :] |= fid[1:, :] != fid[:-1, :]
    edge_img = np.ma.masked_where(~edge, np.ones_like(fid, "float32"))

    fig, ax = plt.subplots(len(seasons), 3, figsize=(15, 4.2 * len(seasons)), squeeze=False)
    R_all = []
    for k, s in enumerate(seasons):
        crop = read_tif(P(f"layers/{s}/cropland.tif")).astype(bool)
        D_lbl = season_window(s)["label"]
        L = {n: read_tif(P(f"layers/{s}/pixel/{n}.tif")) for n in LAYERS}
        M = np.stack([L[n][crop] for n in LAYERS])
        with np.errstate(all="ignore"):
            R = np.corrcoef(M)
        R_all.append(R)
        iu = np.triu_indices(len(LAYERS), 1)
        j = int(np.nanargmax(np.abs(R[iu])))
        print(f"{s}: most redundant pair {LAYERS[iu[0][j]]} / {LAYERS[iu[1][j]]}  r = {R[iu][j]:+.2f}")

        for a, img, ttl in ((ax[k, 0], np.where(crop, L["n_green"], np.nan), "n_green per pixel (days)"),
                            (ax[k, 1], np.where(crop, read_tif(P(f"layers/{s}/n_green.tif")), np.nan),
                             "n_green per CSB field")):
            im = a.imshow(img, cmap="YlGn", vmin=0, vmax=max(1.0, float(np.nanmax(img))))
            a.imshow(edge_img, cmap="gray_r", vmin=0, vmax=1, alpha=0.8, interpolation="nearest")
            a.set_title(f"{s} · {ttl}\nwindow {D_lbl}", fontsize=9); a.set_xticks([]); a.set_yticks([])
            plt.colorbar(im, ax=a, fraction=0.046)
        D = season_window(s)
        inwin = (t >= D["start"]) & (t <= D["end"])
        peak = np.nan_to_num(fused.ndvi.values[inwin], nan=0.0).max(0)[crop]
        ths = np.arange(0.15, 0.61, 0.025)
        ax[k, 2].plot(ths, [(peak > th).mean() for th in ths], "o-", ms=3, color="#8a5a2b")
        ax[k, 2].axvline(CFG["green"], color="#c44", ls="--")
        ax[k, 2].set_title(f"{s} · share of cropland whose NDVI max in the window exceeds x", fontsize=9)
        ax[k, 2].grid(alpha=0.25)
    plt.tight_layout()
    fig_path = os.path.join(FIG_DIR, f"layer_diagnostics_{SITE}.png")
    plt.savefig(fig_path, dpi=120, bbox_inches="tight"); plt.show()

    Rm = np.nanmean(np.stack(R_all), axis=0)
    print(f"\nmean correlation between layers across {len(seasons)} seasons (|r| > 0.9 → drop one):")
    print(f"{'':<11}" + "".join(f"{n:>11}" for n in LAYERS))
    for i, n in enumerate(LAYERS):
        print(f"{n:<11}" + "".join(f"{Rm[i, j]:>11.2f}" for j in range(len(LAYERS))))
    print(f"saved: {fig_path}  (ground-truth fields outlined; speckle = noise, whole fields = signal)")


## Cell 17 — v8 layers on the labelled fields, per season

The value of every v8 layer on each labelled field and season, and the
cover-crop field minus the control within each season (`layer_paired_diff.csv`).
Descriptive: with two fields and three seasons there is no AUC, threshold or test.

In [ ]:
# =============================================================================
# Cell 17 — v8 layers on the labelled fields: values and CC − control per season
# =============================================================================
# Descriptive only (three seasons, two fields): the value of every layer on
# every labelled field and season, and the cover-crop field minus the control
# within each season. No AUC, threshold or test.
for SITE in SITES:
    use_site(SITE)
    labels = pd.read_csv(P("seasons_site.csv"))
    fields = gpd.read_file(P("fields_site_utm.gpkg")).set_index("field_uid")
    core = rxr.open_rasterio(P("field_core.tif")).squeeze(drop=True).values

    # value of every layer on every labelled field × season: mean of the
    # PIXEL layer over the field's core pixels
    rows = []
    for _, r in labels.iterrows():
        d = P(f"layers/{r.season}/pixel")
        if r.field_uid not in fields.index or not os.path.isdir(d):
            continue
        m = core == fields.loc[r.field_uid, "field_id"]
        D = season_window(r.season)
        row = dict(field_uid=r.field_uid, season=r.season, cc_present=int(r.cc_present),
                   window_start=str(D["start"].date()), window_end=str(D["end"].date()),
                   role=fields.loc[r.field_uid, "role"], label_source=r.label_source,
                   label_confidence=r.label_confidence, n_px=int(m.sum()))
        for f in sorted(os.listdir(d)):
            if f.endswith(".tif"):
                row[f[:-4]] = float(read_tif(os.path.join(d, f))[m].mean())
        rows.append(row)
    vals = pd.DataFrame(rows)
    vals.to_csv(P("layer_values_by_season.csv"), index=False)
    names = [n for n in LAYERS + ["et_spring"] if n in vals.columns]

    by_s = vals.groupby(["season", "cc_present"])[names].mean().unstack("cc_present")
    has = lambda s, c: (names[0], c) in by_s.columns and np.isfinite(by_s.loc[s, (names[0], c)])
    both = [s for s in by_s.index if has(s, 1) and has(s, 0)]
    diff = pd.DataFrame({s: {n: by_s.loc[s, (n, 1)] - by_s.loc[s, (n, 0)] for n in names} for s in both})
    diff.index.name = "layer"
    diff.to_csv(P("layer_paired_diff.csv"))
    win = vals.drop_duplicates("season").set_index("season")[["window_start", "window_end"]]
    print(f"{SITE}: v8 layers, CC field − control within each season (descriptive)")
    for s in both:
        print(f"  {s}: window {win.loc[s, 'window_start']} → {win.loc[s, 'window_end']}")
    with pd.option_context("display.float_format", "{:+.3f}".format):
        print(diff.to_string())

    # one panel per layer: the two fields side by side in each season
    fig, ax = plt.subplots(1, len(names), figsize=(2.9 * len(names), 3.8), squeeze=False); ax = ax[0]
    xs = np.arange(len(both))
    for a, n in zip(ax, names):
        for cls, col, off in ((0, "#8a99a5", -0.18), (1, "#2b7a3d", 0.18)):
            v = [by_s.loc[s, (n, cls)] for s in both]
            a.bar(xs + off, v, width=0.34, color=col, label="CC" if cls else "no CC")
        a.set_xticks(xs); a.set_xticklabels(both, fontsize=7, rotation=45)
        a.set_title(n, fontsize=9); a.grid(alpha=0.25, axis="y")
    ax[0].legend(fontsize=7)
    plt.suptitle(f"{SITE}: v8 layers on the labelled fields (core pixels), per season", y=1.03)
    plt.tight_layout()
    fig_path = os.path.join(FIG_DIR, f"layer_eval_{SITE}.png")
    plt.savefig(fig_path, dpi=150, bbox_inches="tight"); plt.show()
    print(f"\nsaved: layer_values_by_season.csv, layer_paired_diff.csv, {fig_path}")


## Cell 18 — Landsat surface temperature

Barnes et al. (2021, Remote Sensing 13:1998) found surface temperature the most
important variable for separating cover crop, residue and bare soil in Indiana
in the fall (bare soil coldest, residue warmest, cover crops in between); NDVI
alone gave κ 0.20. ECOSTRESS covers only 2018 onward and its overpass time
drifts, so this cell adds USGS Landsat Collection 2 Level-2 surface temperature
(Landsat 7/8/9, from Microsoft Planetary Computer): QA_PIXEL first, ST band only
when the ROI is clear enough, reprojected onto the 30 m grid, cached per scene
and mirrored to the Data Store. Relative LST (minus the scene's cropland
median) is computed in Cell 19, once the season's cropland mask is known.

In [ ]:
# =============================================================================
# Cell 18 — Landsat Collection 2 surface temperature onto the 30 m grid
# =============================================================================
# Source: USGS Landsat Collection 2 Level-2 surface temperature (ST_B10 on
# Landsat 8/9, ST_B6 on Landsat 7; 100/60 m thermal resampled by USGS to 30 m)
# from Microsoft Planetary Computer, read window-by-window like HLS.
import pystac_client, planetary_computer

ST_BAD = sum(1 << b for b in range(6))     # QA_PIXEL bits 0-5: fill, dilated cloud,
                                           # cirrus, cloud, shadow, snow
# Planetary Computer is anonymous (signed URLs): no Earthdata header here.
PC_OPTS = dict(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif,.tiff",
               GDAL_HTTP_MAX_RETRY="4", GDAL_HTTP_RETRY_DELAY="3")


def st_search(roi, t0, t1):
    """All Landsat C2 L2 items over the ROI in [t0, t1]."""
    cat = pystac_client.Client.open(CFG["st"]["stac"])
    items = cat.search(collections=[CFG["st"]["collection"]], bbox=list(roi.total_bounds),
                       datetime=f"{t0}/{t1}",
                       query={"eo:cloud_cover": {"lt": CFG["st"]["cloud_max"]},
                              "platform": {"in": CFG["st"]["platforms"]}}).item_collection()
    return list(items)


def fetch_st(item, cache_dir, grid, roi, max_retry=3):
    """One Landsat scene: QA_PIXEL first, surface temperature only if the ROI is clear enough."""
    path = os.path.join(cache_dir, f"ST_{item.id}.nc")
    skip = path[:-3] + ".skip"
    if os.path.exists(path) or os.path.exists(skip):
        return "hit", None
    err = None
    for attempt in range(max_retry):
        try:
            it = planetary_computer.sign(item)          # SAS tokens expire: sign per fetch
            st_key = "lwir11" if "lwir11" in it.assets else "lwir"
            with rasterio.Env(**PC_OPTS):
                qa = open_clip(it.assets["qa_pixel"].href, roi, masked=False).rio.reproject_match(grid)
                q = np.nan_to_num(qa.values, nan=1).astype("uint16")
                clear = (q & ST_BAD) == 0
                if clear.mean() < MIN_CLEAR:
                    with open(skip, "w") as f:
                        f.write(f"clear fraction {clear.mean():.3f}\n")
                    mirror(skip)
                    return "skipped", None
                dn = open_clip(it.assets[st_key].href, roi).rio.reproject_match(grid)  # nodata 0 → NaN
            k = dn * CFG["st"]["scale"] + CFG["st"]["offset"]
            k = k.where(xr.DataArray(clear, coords=k.coords, dims=k.dims) & (k > 200) & (k < 350))
            ds = xr.Dataset({"lst": k.astype("float32")})
            ds["lst"].attrs = {"units": "K"}
            ds["lst"].encoding = {"dtype": "float32", "zlib": True}
            ds.attrs = dict(id=item.id, platform=item.properties["platform"],
                            datetime=str(item.datetime))
            with NC_LOCK:
                ds.to_netcdf(path)
            mirror(path)
            return "new", None
        except Exception as e:
            err = f"{type(e).__name__}: {e}"
            time.sleep(5 * 2 ** attempt)
    return "failed", err


def download_st(items, cache_dir, grid, roi):
    from concurrent.futures import ThreadPoolExecutor, as_completed
    t0 = time.time(); counts = dict(new=0, hit=0, skipped=0, failed=0); failed = []
    ex = ThreadPoolExecutor(N_WORKERS)
    futs = {ex.submit(fetch_st, it, cache_dir, grid, roi): it.id for it in items}
    try:
        for k, f in enumerate(as_completed(futs), 1):
            status, err = f.result(); counts[status] += 1
            if err:
                failed.append(dict(id=futs[f], error=err[:80]))
            if k % 100 == 0:
                print(f"  {k}/{len(items)}  new {counts['new']}  cloud-skipped {counts['skipped']}"
                      f"  ({(time.time()-t0)/60:.1f} min)")
    except KeyboardInterrupt:
        ex.shutdown(wait=False, cancel_futures=True); raise
    ex.shutdown(wait=True)
    print(f"Landsat ST: {counts['new']} new, {counts['skipped']} cloud-skipped, {counts['hit']} cached, "
          f"{counts['failed']} failed ({(time.time()-t0)/60:.1f} min)")
    if failed:
        print(pd.DataFrame(failed).to_string(index=False)); print("Re-run to retry the failures.")


for SITE in SITES:
    use_site(SITE)
    roi = gpd.read_file(P("roi.geojson"))
    grid30 = load_da(P("grid_30m.nc")).rio.write_crs(roi.estimate_utm_crs())
    MIN_CLEAR = 0.05

    have_cache = any(f.endswith(".nc") for f in os.listdir(CACHE_LST))
    if not have_cache and have("lst_scenes.nc"):
        print("per-scene LST cache empty, lst_scenes.nc restored — nothing to download")
        continue
    items = st_search(roi, CFG["date_start"], CFG["date_end"])
    pd.DataFrame([dict(id=i.id, platform=i.properties["platform"], datetime=str(i.datetime),
                       cloud=i.properties.get("eo:cloud_cover")) for i in items]
                 ).to_csv(P("lst_index.csv"), index=False)
    print(f"Landsat C2 L2 scenes over the ROI {CFG['date_start']} → {CFG['date_end']}: {len(items)}  "
          f"{pd.Series([i.properties['platform'] for i in items]).value_counts().to_dict()}")
    download_st(items, CACHE_LST, grid30, roi)

    # stack: one layer per acquisition day (adjacent WRS rows on the same day are averaged)
    ids = {i.id for i in items}                               # only this period's scenes
    files = sorted(f for f in os.listdir(CACHE_LST) if f.endswith(".nc") and f[3:-3] in ids)
    arrs, dates, plats = [], [], []
    for fn in files:
        with xr.open_dataset(os.path.join(CACHE_LST, fn)) as ds:
            a = ds["lst"].values.astype("float32")
            if np.isfinite(a).mean() == 0:
                continue
            arrs.append(a); dates.append(pd.Timestamp(ds.attrs["datetime"][:10]))
            plats.append(ds.attrs["platform"])
    lst = xr.Dataset({"lst": (("time", "y", "x"), np.stack(arrs))},
                     coords={"time": pd.DatetimeIndex(dates), "y": grid30.y.values, "x": grid30.x.values})
    lst = lst.groupby("time").mean("time").sortby("time")
    lst["lst"].attrs = {"units": "K"}
    if os.path.exists(P("lst_scenes.nc")):
        os.remove(P("lst_scenes.nc"))
    lst.to_netcdf(P("lst_scenes.nc"))
    v = lst.lst.values
    print(f"LST stack: {lst.sizes['time']} days  ({pd.Series(plats).value_counts().to_dict()})  "
          f"valid {np.isfinite(v).mean():.0%}  range {np.nanmin(v)-273.15:.1f} … {np.nanmax(v)-273.15:.1f} °C")
    print("saved: lst_index.csv, lst_scenes.nc")


## Cell 18b — Sentinel-1 C-band backscatter

Sentinel-1 IW GRD, radiometrically terrain-corrected gamma0 (VV, VH; 10 m) from
Microsoft Planetary Computer, averaged in linear power onto the 30 m grid and
cached per scene. Radar is the one source with a regular revisit through the
cloudy spring; VH responds to canopy volume, VH − VV partly normalises soil
moisture and roughness. Over IN both relative orbits are ascending (48 and 121);
Sentinel-1B stopped in December 2021, so the three seasons are mostly
Sentinel-1A. Two fields in one scene share the geometry, so Cell 21 also reports
same-day differences.

In [ ]:
# =============================================================================
# Cell 18b — Sentinel-1 C-band backscatter (VV, VH) onto the 30 m grid
# =============================================================================
# Source: Sentinel-1 IW GRD, radiometrically terrain corrected (gamma0, linear
# power, 10 m) from Microsoft Planetary Computer ("sentinel-1-rtc"). Radar
# sees through cloud, so it is the only source with a regular revisit in the
# cloudy spring window. VH (cross-pol) responds to volume scattering from a
# canopy; VH − VV (dB) normalises part of the soil-moisture and roughness effect.
# Each scene is averaged in LINEAR power onto the 30 m grid (speckle falls as
# ~1/√n with the 9 pixels per cell), cached per scene and mirrored. Scenes of
# different relative orbits see the fields from different angles: compare the
# two fields within a scene (they share the geometry), and keep the orbit in
# the stack so it can be separated later.
from rasterio.enums import Resampling
S1_NODATA = -32768.0


def s1_search(roi, t0, t1):
    cat = pystac_client.Client.open(CFG["s1"]["stac"])
    items = cat.search(collections=[CFG["s1"]["collection"]], bbox=list(roi.total_bounds),
                       datetime=f"{t0}/{t1}").item_collection()
    return list(items)


def fetch_s1(item, cache_dir, grid, roi, max_retry=3):
    path = os.path.join(cache_dir, f"S1_{item.id}.nc")
    if os.path.exists(path):
        return "hit", None
    err = None
    for attempt in range(max_retry):
        try:
            it = planetary_computer.sign(item)
            ds = xr.Dataset()
            with rasterio.Env(**PC_OPTS):
                for pol in ("vv", "vh"):
                    da = open_clip(it.assets[pol].href, roi, masked=False).astype("float32")
                    da = da.where((da > 0) & (da != S1_NODATA))
                    da = da.rio.write_nodata(np.nan).rio.reproject_match(
                        grid, resampling=Resampling.average)
                    ds[pol] = da.astype("float32")
                    ds[pol].attrs = {"units": "gamma0, linear power"}
                    ds[pol].encoding = {"dtype": "float32", "zlib": True}
            if not np.isfinite(ds["vh"].values).any():
                return "empty", None
            ds.attrs = dict(id=item.id, platform=item.properties.get("platform", ""),
                            datetime=str(item.datetime),
                            orbit_state=item.properties.get("sat:orbit_state", ""),
                            relative_orbit=int(item.properties.get("sat:relative_orbit", -1)))
            with NC_LOCK:
                ds.to_netcdf(path)
            mirror(path)
            return "new", None
        except Exception as e:
            err = f"{type(e).__name__}: {e}"
            time.sleep(5 * 2 ** attempt)
    return "failed", err


def download_s1(items, cache_dir, grid, roi):
    from concurrent.futures import ThreadPoolExecutor, as_completed
    t0 = time.time(); counts = dict(new=0, hit=0, empty=0, failed=0); failed = []
    ex = ThreadPoolExecutor(N_WORKERS)
    futs = {ex.submit(fetch_s1, it, cache_dir, grid, roi): it.id for it in items}
    try:
        for k, f in enumerate(as_completed(futs), 1):
            status, err = f.result(); counts[status] += 1
            if err:
                failed.append(dict(id=futs[f], error=err[:80]))
            if k % 50 == 0:
                print(f"  {k}/{len(items)}  new {counts['new']}  ({(time.time()-t0)/60:.1f} min)")
    except KeyboardInterrupt:
        ex.shutdown(wait=False, cancel_futures=True); raise
    ex.shutdown(wait=True)
    print(f"Sentinel-1: {counts['new']} new, {counts['hit']} cached, {counts['empty']} outside the ROI, "
          f"{counts['failed']} failed ({(time.time()-t0)/60:.1f} min)")
    if failed:
        print(pd.DataFrame(failed).to_string(index=False)); print("Re-run to retry the failures.")


for SITE in SITES:
    use_site(SITE)
    roi = gpd.read_file(P("roi.geojson"))
    grid30 = load_da(P("grid_30m.nc")).rio.write_crs(roi.estimate_utm_crs())

    have_cache = any(f.endswith(".nc") for f in os.listdir(CACHE_S1))
    if not have_cache and have("s1_scenes.nc"):
        print("per-scene Sentinel-1 cache empty, s1_scenes.nc restored — nothing to download")
        continue
    items = s1_search(roi, CFG["date_start"], CFG["date_end"])
    idx = pd.DataFrame([dict(id=i.id, platform=i.properties.get("platform"), datetime=str(i.datetime),
                             orbit_state=i.properties.get("sat:orbit_state"),
                             relative_orbit=i.properties.get("sat:relative_orbit")) for i in items])
    idx.to_csv(P("s1_index.csv"), index=False)
    print(f"Sentinel-1 RTC scenes over the ROI {CFG['date_start']} → {CFG['date_end']}: {len(items)}  "
          f"{idx.groupby(['platform', 'orbit_state', 'relative_orbit']).size().to_dict()}")
    download_s1(items, CACHE_S1, grid30, roi)

    # stack the scenes of THIS period (the cache may hold other periods)
    ids = set(idx.id)
    vv, vh, dates, orbits = [], [], [], []
    for fn in sorted(os.listdir(CACHE_S1)):
        if not fn.endswith(".nc") or fn[3:-3] not in ids:
            continue
        with xr.open_dataset(os.path.join(CACHE_S1, fn)) as ds:
            vv.append(ds["vv"].values.astype("float32")); vh.append(ds["vh"].values.astype("float32"))
            dates.append(pd.Timestamp(ds.attrs["datetime"][:10])); orbits.append(int(ds.attrs["relative_orbit"]))
    s1 = xr.Dataset({"vv": (("time", "y", "x"), np.stack(vv)), "vh": (("time", "y", "x"), np.stack(vh))},
                    coords={"time": pd.DatetimeIndex(dates), "y": grid30.y.values, "x": grid30.x.values})
    # two slices of one pass on the same day are the same acquisition: average them
    s1 = s1.groupby("time").mean("time").sortby("time")
    orb = pd.Series(orbits, index=pd.DatetimeIndex(dates)).groupby(level=0).first()
    s1 = s1.assign_coords(relative_orbit=("time", orb.reindex(s1.time.values).values.astype("int16")))
    for v in ("vv", "vh"):
        s1[v].attrs = {"units": "gamma0, linear power"}
    if os.path.exists(P("s1_scenes.nc")):
        os.remove(P("s1_scenes.nc"))
    with NC_LOCK:
        s1.to_netcdf(P("s1_scenes.nc"))
    with np.errstate(all="ignore"):
        vh_db = 10 * np.log10(s1.vh.values)
    print(f"S1 stack: {s1.sizes['time']} days, relative orbits {sorted(set(orbits))}, "
          f"VH {np.nanpercentile(vh_db, 5):.1f} … {np.nanpercentile(vh_db, 95):.1f} dB (5-95 %)")
    print("saved: s1_index.csv, s1_scenes.nc, cache/s1/")


## Cell 18c — PlanetScope through NASA CSDA

Every run lists the PlanetScope scenes CSDA holds over the ROI (public STAC,
no login; `ps_index.csv`) and counts the 8-band scenes per season and window.
Download needs the CSDA authorization: put the Earthdata login in `~/.netrc`,
set `CFG["ps"]["enabled"] = True`, and re-run this cell and the ones below.
Each scene's 8-band surface reflectance and UDM2 clear mask are read for the ROI
only (signed URL window read when the API redirects, else download-and-clip),
put on a 3 m grid, cross-calibrated band by band to the HLS scene of the same
or neighbouring day (blue, green, red, NIR; the 705 nm red edge has no HLS
counterpart here), and averaged per field (`ps_unit_obs.csv`: NDVI, GCVI, NDRE).

In [ ]:
# =============================================================================
# Cell 18c — PlanetScope (NASA CSDA): inventory now, download once authorized
# =============================================================================
# Search is public: every run lists the PlanetScope scenes that CSDA holds over
# the ROI (ps_index.csv) and counts them per season and window. Download needs
# CSDA authorization; until then CFG["ps"]["enabled"] stays False and the cell
# stops after the inventory. Once authorized:
#   1. put the Earthdata login in ~/.netrc (machine urs.earthdata.nasa.gov)
#   2. set CFG["ps"]["enabled"] = True and re-run this cell and the ones below.
# Per scene: 8-band surface reflectance (SuperDove; 4-band scenes are skipped)
# and the UDM2 clear mask, clipped to the ROI on a 3 m UTM grid and cached as
# int16 NetCDF (cache/ps, mirrored). Each scene is cross-calibrated band by
# band (blue, green, red, NIR) to the HLS scene of the same or neighbouring day
# over clear pixels aggregated to 30 m; the red edge (705 nm) has no HLS
# counterpart in this cache and stays uncalibrated. Field means per scene go
# to ps_unit_obs.csv (ground-truth fields and CSB fields) for Cells 20-21.
import io, netrc, requests
from urllib.parse import urlparse, parse_qs
from rasterio.enums import Resampling

PS_BANDS = ["coastal_blue", "blue", "green_i", "green", "yellow", "red", "rededge", "nir"]
PS_TO_HLS = {"blue": "blue", "green": "green", "red": "red", "nir": "nir"}


def ps_stac_search(roi, t0, t1):
    """All CSDA PlanetScope items over the ROI in [t0, t1] (public STAC, no login)."""
    geom = json.loads(gpd.GeoSeries([roi.to_crs(4326).unary_union], crs=4326).to_json())["features"][0]["geometry"]
    url, body, out = CFG["ps"]["stac"].rstrip("/") + "/search", dict(
        collections=[CFG["ps"]["collection"]], intersects=geom, limit=250,
        datetime=f"{pd.Timestamp(t0):%Y-%m-%dT00:00:00Z}/{pd.Timestamp(t1):%Y-%m-%dT23:59:59Z}"), []
    while url:
        r = requests.post(url, json=body, timeout=120) if body is not None else requests.get(url, timeout=120)
        r.raise_for_status(); j = r.json()
        out += j.get("features", [])
        nxt = [l for l in j.get("links", []) if l.get("rel") == "next"]
        url = nxt[0]["href"] if nxt else None
        body = (nxt[0].get("body") or body) if nxt and nxt[0].get("method", "GET") == "POST" else None
    return out


def ps_window(d):
    return "fall" if d.month in (10, 11, 12) else "spring" if d.month in (3, 4, 5) else "other"


def csda_session():
    """Log in to the CSDA API with the Earthdata credentials in ~/.netrc (same
    flow as NASA-IMPACT/csda-client, which needs Python ≥ 3.12)."""
    api = CFG["ps"]["api"].rstrip("/")
    auth = netrc.netrc().authenticators("urs.earthdata.nasa.gov")
    if not auth:
        raise RuntimeError("no Earthdata entry in ~/.netrc (machine urs.earthdata.nasa.gov)")
    s = requests.Session(); s.headers["User-Agent"] = "CoverCrop_Fusion"
    r = s.get(f"{api}/api/v1/auth/", params={"redirect_uri": api}, allow_redirects=False, timeout=60)
    if r.status_code not in (302, 307):
        raise RuntimeError(f"CSDA auth: expected a redirect, got {r.status_code}")
    r = s.get(r.headers["Location"], auth=(auth[0], auth[2]), allow_redirects=False, timeout=60)
    q = parse_qs(urlparse(r.headers.get("Location", "")).query)
    if "code" not in q:
        raise RuntimeError("Earthdata did not return a code — is the CSDA application authorized for "
                           "this account, and is the CSDA program authorization approved?")
    tok = s.post(f"{api}/api/v1/auth/token", data={"code": q["code"][0]}, timeout=60).json()["access_token"]
    s.headers["Authorization"] = f"Bearer {tok}"
    s.verify_msg = s.get(f"{api}/api/v1/auth/verify", timeout=60).text[:200]
    return s


def ps_read_asset(sess, item_id, asset, roi):
    """(bands, H, W) array of one asset clipped to the ROI, with its transform
    and CRS. A redirect to a signed URL is read window-by-window (no full
    download); otherwise the file is streamed to a temporary file and clipped."""
    api = CFG["ps"]["api"].rstrip("/")
    url = f"{api}/api/v2/download/{CFG['ps']['collection']}/{item_id}/{asset}"
    r = sess.get(url, allow_redirects=False, timeout=120)
    tmp = None
    if r.status_code in (301, 302, 303, 307, 308):
        src_path = "/vsicurl/" + r.headers["Location"]
    else:
        r = sess.get(url, stream=True, timeout=600); r.raise_for_status()
        tmp = os.path.join(CACHE_PS, f"_{item_id}_{asset}.tif")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
        src_path = tmp
    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR"), rasterio.open(src_path) as src:
            bb = roi.to_crs(src.crs).total_bounds
            win = (rasterio.windows.from_bounds(*bb, transform=src.transform).round_offsets()
                   .round_lengths().intersection(rasterio.windows.Window(0, 0, src.width, src.height)))
            arr = src.read(window=win)
            return arr, src.window_transform(win), src.crs
    finally:
        if tmp and os.path.exists(tmp):
            os.remove(tmp)


def ps_scene_to_grid(sr, udm, tr, crs, grid3):
    """8-band SR (DN = reflectance x 10 000, 0 = no data) + UDM2 → Dataset on
    the 3 m grid, int16 with −9999 where not clear."""
    def to_da(a):
        ny, nx = a.shape
        return (xr.DataArray(a, dims=("y", "x"),
                             coords={"y": tr.f + tr.e * (np.arange(ny) + 0.5), "x": tr.c + tr.a * (np.arange(nx) + 0.5)})
                .rio.write_crs(crs).rio.write_transform(tr))
    clear = to_da((udm[0] == 1).astype("uint8")).rio.write_nodata(0).rio.reproject_match(
        grid3, resampling=Resampling.nearest).values == 1
    ds = xr.Dataset()
    for k, b in enumerate(PS_BANDS):
        a = sr[k].astype("float32"); a[a <= 0] = np.nan
        v = to_da(a).rio.write_nodata(np.nan).rio.reproject_match(grid3, resampling=Resampling.nearest).values
        v = np.where(clear & np.isfinite(v), v, np.nan)
        ds[b] = (("y", "x"), v.astype("float32"))
        ds[b].encoding = {"dtype": "int16", "_FillValue": -9999, "zlib": True}
    return ds.assign_coords(y=grid3.y.values, x=grid3.x.values)


def make_grid3(grid30):
    """3 m grid covering the 30 m grid exactly (10 x 10 cells per 30 m cell)."""
    r = CFG["ps"]["res"]; f = int(round(CFG["res_optical"] / r))
    x30, y30 = grid30.x.values, grid30.y.values
    dx, dy = x30[1] - x30[0], y30[1] - y30[0]
    xs = (x30[:, None] - dx / 2 + (np.arange(f) + 0.5) * dx / f).ravel()
    ys = (y30[:, None] - dy / 2 + (np.arange(f) + 0.5) * dy / f).ravel()
    g = xr.DataArray(np.zeros((len(ys), len(xs)), "uint8"), coords={"y": ys, "x": xs}, dims=("y", "x"))
    return g.rio.write_crs(grid30.rio.crs)


for SITE in SITES:
    use_site(SITE)
    roi = gpd.read_file(P("roi.geojson"))
    grid30 = load_da(P("grid_30m.nc")).rio.write_crs(roi.estimate_utm_crs())

    # --- 1. inventory (public, always) --------------------------------------
    try:
        feats = ps_stac_search(roi, CFG["date_start"], CFG["date_end"])
    except Exception as e:
        feats = None
        print(f"CSDA STAC search failed ({type(e).__name__}: {str(e)[:120]}) — keeping the previous ps_index.csv")
    if feats is not None:
        inv = pd.DataFrame([dict(id=f["id"], datetime=f["properties"]["datetime"],
                                 instrument=f["properties"].get("eo:instrument"),
                                 item_type=f["properties"].get("pl:item_type"),
                                 clear_percent=f["properties"].get("pl:clear_percent"),
                                 sr8=CFG["ps"]["sr_asset"] in f["assets"],
                                 udm2=CFG["ps"]["udm_asset"] in f["assets"]) for f in feats])
        if len(inv):
            d = pd.to_datetime(inv.datetime.str[:19])
            inv["date"] = d.dt.date.astype(str)
            y0 = np.where(d.dt.month >= 9, d.dt.year, d.dt.year - 1)
            inv["season"] = [f"{a}-{str(a + 1)[2:]}" for a in y0]
            inv["window"] = [ps_window(x) for x in d]
            inv = inv.sort_values("datetime")
            inv.to_csv(P("ps_index.csv"), index=False)
    inv = pd.read_csv(P("ps_index.csv")) if have("ps_index.csv") else pd.DataFrame()
    labels = pd.read_csv(P("seasons_site.csv"))
    if len(inv):
        inv = inv[inv.season.isin(labels.season.unique())]
        use = inv[inv.sr8 & inv.udm2 & (inv.clear_percent >= CFG["ps"]["min_clear_percent"])]
        tab = (inv.groupby(["season", "window"]).size().rename("all").to_frame()
               .join(use.groupby(["season", "window"]).size().rename(f"8-band, clear ≥ {CFG['ps']['min_clear_percent']}%"))
               .fillna(0).astype(int))
        print(f"CSDA PlanetScope over the ROI, labelled seasons: {len(inv)} scenes "
              f"({inv.instrument.value_counts().to_dict()})")
        print(tab.to_string())
        for (s, w), g in use.groupby(["season", "window"]):
            if w != "other":
                print(f"  {s} {w:<6}: " + ", ".join(sorted(set(g.date.str[5:]))))
    else:
        use = pd.DataFrame()
        print("no CSDA PlanetScope scenes over the ROI")

    # --- 2. download + process (needs CSDA authorization) ----------------------
    if not CFG["ps"]["enabled"]:
        print("\nPlanetScope download is OFF (CFG['ps']['enabled'] = False): waiting for CSDA "
              "authorization. Cells 20-21 run without PlanetScope.")
        continue
    grid3 = make_grid3(grid30)
    sess = ps_read = None
    todo = [r for _, r in use.iterrows() if r.window != "other"
            and not os.path.exists(os.path.join(CACHE_PS, f"PS_{r.id}.nc"))]
    print(f"\nPlanetScope: {len(use)} usable scenes, {len(todo)} to fetch")
    if todo:
        sess = csda_session()
        print(f"CSDA login OK ({sess.verify_msg[:60]})")
    for k, r in enumerate(todo, 1):
        try:
            sr, tr, crs = ps_read_asset(sess, r.id, CFG["ps"]["sr_asset"], roi)
            udm, tr_u, crs_u = ps_read_asset(sess, r.id, CFG["ps"]["udm_asset"], roi)
            if sr.shape[0] != 8 or sr.shape[1:] != udm.shape[1:]:
                print(f"  {r.id}: unexpected shapes {sr.shape} / {udm.shape} — skipped"); continue
            ds = ps_scene_to_grid(sr, udm, tr, crs, grid3)
            ds.attrs = dict(id=r.id, datetime=str(r.datetime), instrument=str(r.instrument))
            path = os.path.join(CACHE_PS, f"PS_{r.id}.nc")
            with NC_LOCK:
                ds.to_netcdf(path)
            mirror(path)
        except Exception as e:
            print(f"  {r.id}: {type(e).__name__}: {str(e)[:120]}")
        if k % 10 == 0:
            print(f"  {k}/{len(todo)}")

    # --- 3. cross-calibration to HLS, field means ------------------------------
    with xr.open_dataset(P("hls_bands.nc")) as _b:
        t_h = pd.DatetimeIndex(_b.time.values)
        HB = {b: _b[b].values for b in PS_TO_HLS.values()}
    fields = gpd.read_file(P("fields_site_utm.gpkg"))
    csb = gpd.read_file(P("csb_roi.gpkg")).to_crs(grid3.rio.crs).reset_index(drop=True)
    tr3, shape3 = grid3.rio.transform(), grid3.shape
    inset = -CFG["ps"]["edge_m"]
    gt_lab = features.rasterize(zip(fields.geometry.buffer(inset), fields.field_id), out_shape=shape3,
                                transform=tr3, fill=0, dtype="int32")
    csb_lab = features.rasterize(zip(csb.geometry.buffer(inset), np.arange(1, len(csb) + 1)), out_shape=shape3,
                                 transform=tr3, fill=0, dtype="int32")
    units = [("gt_field", gt_lab, dict(zip(fields.field_id, fields.field_uid))),
             ("csb_field", csb_lab, dict(zip(range(1, len(csb) + 1), csb.CSBID.astype(str))))]
    cal_rows, obs_rows = [], []
    ids = set(use.id)
    for fn in sorted(os.listdir(CACHE_PS)):
        if not fn.endswith(".nc") or fn[3:-3] not in ids:
            continue
        with xr.open_dataset(os.path.join(CACHE_PS, fn)) as ds:
            B = {b: ds[b].values.astype("float32") / 1e4 for b in PS_BANDS}
            day = pd.Timestamp(ds.attrs["datetime"][:10]); pid = ds.attrs["id"]
        near = np.where(np.abs((t_h - day).days) <= CFG["ps"]["match_days"])[0]
        coef = {}
        for pb, hb in PS_TO_HLS.items():
            a30 = (xr.DataArray(B[pb], coords=grid3.coords, dims=grid3.dims).rio.write_crs(grid3.rio.crs)
                   .rio.write_nodata(np.nan).rio.reproject_match(grid30, resampling=Resampling.average).values)
            best = None
            for i in near:
                h = HB[hb][i]; ok = np.isfinite(a30) & np.isfinite(h)
                if ok.sum() >= CFG["ps"]["min_cal_px"]:
                    slope, icpt = np.polyfit(a30[ok], h[ok], 1)
                    rr = np.corrcoef(a30[ok], h[ok])[0, 1]
                    if best is None or rr > best[2]:
                        best = (slope, icpt, rr, int(ok.sum()), str(t_h[i].date()))
            if best and best[2] >= CFG["ps"]["min_cal_r"]:
                coef[pb] = best[:2]
                cal_rows.append(dict(id=pid, date=str(day.date()), band=pb, slope=best[0], intercept=best[1],
                                     r=best[2], n_px=best[3], hls_date=best[4]))
        C = {b: (coef[b][0] * B[b] + coef[b][1]) if b in coef else B[b] for b in PS_BANDS}
        with np.errstate(all="ignore"):
            IDX = {"ndvi": (C["nir"] - C["red"]) / (C["nir"] + C["red"]),
                   "gcvi": C["nir"] / C["green"] - 1,
                   "ndre": (C["nir"] - C["rededge"]) / (C["nir"] + C["rededge"])}
        calibrated = all(b in coef for b in ("red", "nir", "green"))
        for unit, lab, names in units:
            lf = lab.ravel(); n_u = len(names)
            tot = np.bincount(lf, minlength=n_u + 1)[1:].astype(float)
            clear = np.isfinite(C["red"]).ravel()
            nclr = np.bincount(lf[clear], minlength=n_u + 1)[1:]
            res = {}
            for k_, v in IDX.items():
                vv = v.ravel(); ok = np.isfinite(vv)
                s = np.bincount(lf[ok], weights=vv[ok], minlength=n_u + 1)[1:]
                n = np.bincount(lf[ok], minlength=n_u + 1)[1:]
                with np.errstate(all="ignore"):
                    res[k_] = s / n
            for j, (uid_i, uid) in enumerate(names.items()):
                if tot[j] == 0:
                    continue
                obs_rows.append(dict(unit=unit, uid=uid, date=str(day.date()), id=pid,
                                     clear=nclr[j] / tot[j], calibrated=calibrated,
                                     **{k_: res[k_][j] for k_ in IDX}))
    pd.DataFrame(cal_rows).to_csv(P("ps_calibration.csv"), index=False)
    obs = pd.DataFrame(obs_rows)
    obs.to_csv(P("ps_unit_obs.csv"), index=False)
    if len(obs):
        g = obs[obs.unit == "gt_field"]
        print(f"PlanetScope field observations: {g.date.nunique()} days on the labelled fields, "
              f"{int(g.calibrated.mean() * 100)}% of scenes calibrated to HLS")
    print("saved: ps_index.csv, ps_calibration.csv, ps_unit_obs.csv, cache/ps/")


## Cell 19 — Daymet weather (the other 16 predictors of the Ma et al. baseline)

Ma et al. (2026) pair the 128 Landsat features with each field's **monthly mean
temperature and monthly accumulated precipitation, Oct–May** (18 features × 8
months = 144). The paper does not name its weather product; this uses
**Daymet V4 R1** (1 km daily, ORNL DAAC) through the single-pixel API at each
unit's centroid — the labelled fields and every CSB field — cached in
`cache/daymet/` and mirrored. Units in the same 1 km Daymet pixel get identical
series and share one pixel id.

Within one site the weather is (nearly) the same for every field, so it cannot
separate a cover-crop field from its neighbour in the same season; it matters
once sites and years are pooled, where it tells the model what kind of winter
the spectral features come from.

In [ ]:
# =============================================================================
# Cell 19 — Daymet daily weather for every unit (Ma et al. weather covariates)
# =============================================================================
# Ma et al. (2026) add, for each field, the monthly mean temperature and the
# monthly accumulated precipitation of Oct–May to the 128 Landsat features
# (18 features x 8 months = 144). The paper does not name the weather product;
# here: Daymet V4 R1 (1 km, daily; ORNL DAAC, doi:10.3334/ORNLDAAC/2129) through
# its single-pixel API, queried at each unit's centroid (cached as CSV, mirrored).
# Units that fall in the same 1 km Daymet pixel return identical series and are
# given one pixel id. (Do not snap centroids to a coarser lat/lon grid: v10's
# first run did, and moved one of two fields 300 m apart into the neighbouring
# pixel — a fake, perfectly consistent weather "difference" between them.)
# Daymet uses a 365-day year: in leap years Dec 31 is not provided.
import requests, io, hashlib


def wx_key(lat, lon):
    rd = CFG["wx"]["round_deg"]
    return f"{round(lat / rd) * rd:.4f}_{round(lon / rd) * rd:.4f}"


def daymet_point(key, start, end, cache_dir, max_retry=3):
    """Daily tmax, tmin, prcp for one location (cached); columns date, tmean, prcp."""
    path = os.path.join(cache_dir, f"daymet_{key}.csv")
    if os.path.exists(path):
        d = pd.read_csv(path, parse_dates=["date"])
        if d.date.min() <= pd.Timestamp(start) + pd.Timedelta(days=1):
            return d, "cached"
    lat, lon = map(float, key.split("_"))
    err = None
    for attempt in range(max_retry):
        try:
            r = requests.get(CFG["wx"]["api"], timeout=120,
                             params=dict(lat=lat, lon=lon, vars=CFG["wx"]["vars"], start=start, end=end))
            r.raise_for_status()
            lines = r.text.splitlines()
            h = next(i for i, l in enumerate(lines) if l.startswith("year,yday"))
            d = pd.read_csv(io.StringIO("\n".join(lines[h:])))
            d.columns = [c.split(" ")[0] for c in d.columns]          # "prcp (mm/day)" → "prcp"
            d["date"] = pd.to_datetime(d.year.astype(str) + d.yday.astype(str).str.zfill(3), format="%Y%j")
            d["tmean"] = (d.tmax + d.tmin) / 2
            d = d[["date", "tmax", "tmin", "tmean", "prcp"]]
            d.to_csv(path, index=False)
            mirror(path)
            return d, "new"
        except Exception as e:
            err = f"{type(e).__name__}: {e}"
            time.sleep(5 * 2 ** attempt)
    raise RuntimeError(f"Daymet request failed for {key}: {err}")


for SITE in SITES:
    use_site(SITE)
    fields = gpd.read_file(P("fields_site_utm.gpkg"))
    csb = gpd.read_file(P("csb_roi.gpkg")).reset_index(drop=True)
    units = []
    for unit, uids, geom in (("gt_field", fields.field_uid.astype(str), fields.geometry),
                             ("csb_field", csb.CSBID.astype(str), csb.geometry)):
        cen = gpd.GeoSeries(geom.centroid, crs=geom.crs).to_crs(4326)   # centroid in a projected CRS
        units += [dict(unit=unit, uid=u, lat=p.y, lon=p.x, key=wx_key(p.y, p.x)) for u, p in zip(uids, cen)]
    units = pd.DataFrame(units)

    start = str(pd.Timestamp(CFG["date_start"]).date()); end = str(pd.Timestamp(CFG["date_end"]).date())
    frames, status, pixel = [], [], {}
    for q in sorted(units.key.unique()):
        d, s = daymet_point(q, start, end, CACHE_WX)
        status.append(s)
        pid = "px_" + hashlib.md5(d[["tmax", "tmin", "prcp"]].round(2).to_numpy().tobytes()).hexdigest()[:10]
        pixel[q] = pid
        if pid not in {f.key.iloc[0] for f in frames}:
            frames.append(d.assign(key=pid))
    units = units.rename(columns={"key": "query"}).assign(key=lambda u: u["query"].map(pixel))
    units.to_csv(P("unit_weather.csv"), index=False)
    wx = pd.concat(frames, ignore_index=True)
    wx.to_csv(P("daymet_daily.csv"), index=False)

    gk = units[units.unit == "gt_field"].set_index("uid").key
    print(f"Daymet V4 R1: {units['query'].nunique()} centroid request(s) → {units.key.nunique()} distinct 1 km pixel(s) "
          f"for {len(units)} units ({status.count('new')} new, {status.count('cached')} cached); "
          f"coverage {wx.date.min().date()} → {wx.date.max().date()}")
    print(f"ground-truth fields: {gk.to_dict()}"
          + ("  — same Daymet pixel: within a season their weather features are identical"
             if gk.nunique() == 1 else ""))
    # season overview for the ground-truth location(s)
    rows = []
    for season in sorted(pd.read_csv(P("seasons_site.csv")).season.unique()):
        D = season_window(season)
        w0 = pd.Timestamp(CFG["feat"]["window"][0].replace("y0", str(D["y0"])).replace("y1", str(D["y1"])))
        w1 = pd.Timestamp(CFG["feat"]["window"][1].replace("y0", str(D["y0"])).replace("y1", str(D["y1"])))
        s = wx[wx.key.isin(gk.unique()) & (wx.date >= w0) & (wx.date <= w1)]
        rows.append(dict(season=season, days=s.date.nunique(), tmean_OctMay=s.tmean.mean(),
                         prcp_OctMay=s.groupby("key").prcp.sum().mean(),
                         tmean_Nov=s[s.date.dt.month == 11].tmean.mean(),
                         tmean_Mar=s[s.date.dt.month == 3].tmean.mean()))
    print(pd.DataFrame(rows).round(1).to_string(index=False))
    print("saved: unit_weather.csv, daymet_daily.csv, cache/daymet/")


## Cell 20 — Season feature table

For each season (fall year y0, spring year y1) and each unit — the labelled
fields (core pixels) and every CSB field ≥ `min_field_px` — from **observed
scenes only** (a unit's scene value is kept when ≥ `min_clear` of it is clear):

| group | features | source |
|---|---|---|
| Ma et al. baseline | blue, green, red, NIR, SWIR1, SWIR2, NDVI, GCVI: 2nd-order harmonic fit over Oct 1 y0 – May 31 y1, min and max of the fitted curve per month (8 × 8 × 2 = 128) | Ma, Shen, Swatantran & Lobell 2026, IJAEOG 149:105258 |
| extension | NDTI and STI the same way (32); relative LST monthly mean (8) | Barnes et al. 2021 indices |
| raw monthly max | NDVI, GCVI, NDTI: monthly max of real observations, no harmonic (24) | guards against the harmonic smoothing out a two-week spring signal |
| Barnes fall | mean NDVI, NDTI, STI, relative LST over Oct 1 – Dec 1 of y0 | Barnes et al. 2021 |
| spring | max NDVI Apr 1 – May 31 of y1 | |

| Sentinel-1 | monthly mean VV, VH and VH − VV (dB) of the unit, Oct–May (8 × 3 = 24), from Cell 18b | C-band backscatter |
| Ma et al. weather | monthly mean temperature and monthly total precipitation, Oct–May (8 × 2 = 16), from Cell 19 (Daymet) | Ma et al. 2026 |

With the weather the Ma et al. baseline is the paper's 144 predictors. Five per-pixel layers per season go to `layers/<season>/` for firerx
(fall NDVI/NDTI/STI/relative LST, spring NDVI max); a pixel with no clear
observation gets the season's cropland median, and non-cropland is 0.

In [ ]:
# =============================================================================
# Cell 20 — Season feature table: Ma et al. 144 + NDTI/STI + LST + Sentinel-1 + fall window
# =============================================================================
# One row per unit (ground-truth field, and every CSB field) per season. Units
# are summarised scene by scene from REAL observations only (no gap filling):
# a unit's value in a scene is the mean of its clear pixels, kept only if at
# least `min_clear` of the unit is clear.
BANDS6 = ["blue", "green", "red", "nir", "swir1", "swir2"]
MA_VARS = BANDS6 + ["ndvi", "gcvi"]          # Ma et al. 2026
EXTRA = ["ndti", "sti"]                      # Barnes et al. 2021 residue indices
MON = CFG["feat"]["months"]


def unit_means(stack, lab, n_units, min_clear):
    """stack (T, Y, X) → (T, U) unit means over clear pixels; NaN when too cloudy."""
    lab_f = lab.ravel()
    tot = np.bincount(lab_f, minlength=n_units + 1)[1:].astype(float)
    out = np.full((stack.shape[0], n_units), np.nan, "float32")
    for i in range(stack.shape[0]):
        v = stack[i].ravel(); ok = np.isfinite(v)
        s = np.bincount(lab_f[ok], weights=v[ok], minlength=n_units + 1)[1:]
        n = np.bincount(lab_f[ok], minlength=n_units + 1)[1:]
        with np.errstate(all="ignore"):
            m = s / n
        m[(n / np.maximum(tot, 1)) < min_clear] = np.nan
        out[i] = m
    return out


def harmonic_month_minmax(tdays, y, month_of_day, order, min_obs):
    """Fit y(t) = a0 + Σ_k a_k cos(2πkt/365.25) + b_k sin(2πkt/365.25) to the
    observed points; evaluate daily; return (min, max) per calendar month in MON."""
    ok = np.isfinite(y)
    if ok.sum() < max(min_obs, 2 * order + 1):
        return np.full(len(MON), np.nan), np.full(len(MON), np.nan)
    def design(t):
        cols = [np.ones_like(t, float)]
        for k in range(1, order + 1):
            w = 2 * np.pi * k * t / 365.25
            cols += [np.cos(w), np.sin(w)]
        return np.stack(cols, 1)
    coef, *_ = np.linalg.lstsq(design(tdays[ok]), y[ok], rcond=None)
    days = np.arange(len(month_of_day)); fit = design(days) @ coef
    mn = np.array([fit[month_of_day == m].min() for m in MON])
    mx = np.array([fit[month_of_day == m].max() for m in MON])
    return mn, mx


def season_rows(units_name, lab, uids, D, t_opt, VARS, t_lst, LSTREL, extra_cols):
    """Feature rows for every unit of one season."""
    ts = lambda s: pd.Timestamp(s.replace("y0", str(D["y0"])).replace("y1", str(D["y1"])))
    w0, w1 = ts(CFG["feat"]["window"][0]), ts(CFG["feat"]["window"][1])
    f0, f1 = ts(CFG["feat"]["fall"][0]), ts(CFG["feat"]["fall"][1])
    s0, s1 = ts(CFG["feat"]["spring_max"][0]), ts(CFG["feat"]["spring_max"][1])
    days_idx = pd.date_range(w0, w1, freq="D")
    month_of_day = days_idx.month.values
    sel = (t_opt >= w0) & (t_opt <= w1)
    tdays = (t_opt[sel] - w0).days.values.astype(float)
    tmon = t_opt[sel].month.values
    fall = (t_opt >= f0) & (t_opt < f1); spr = (t_opt >= s0) & (t_opt <= s1)
    lsel = (t_lst >= w0) & (t_lst <= w1); lmon = t_lst[lsel].month.values
    lfall = (t_lst >= f0) & (t_lst < f1)
    rows = []
    for j, uid in enumerate(uids):
        r = dict(unit=units_name, uid=uid, season=D["season"], **extra_cols.get(uid, {}))
        # Ma et al.: harmonic monthly min/max of 6 bands + NDVI + GCVI (+ NDTI, STI)
        for v in MA_VARS + EXTRA:
            mn, mx = harmonic_month_minmax(tdays, VARS[v][sel, j], month_of_day,
                                           CFG["feat"]["harm_order"], CFG["feat"]["min_obs"])
            for m, a, b in zip(MON, mn, mx):
                r[f"{v}_m{m:02d}_min"] = a; r[f"{v}_m{m:02d}_max"] = b
        # the same without the harmonic: monthly max of real observations only
        for v in ("ndvi", "gcvi", "ndti"):
            y = VARS[v][sel, j]
            for m in MON:
                yy = y[tmon == m]
                r[f"raw_{v}_m{m:02d}_max"] = np.nanmax(yy) if np.isfinite(yy).any() else np.nan
        # surface temperature relative to the scene's cropland median, monthly mean
        yl = LSTREL[lsel, j]
        for m in MON:
            yy = yl[lmon == m]
            r[f"lstrel_m{m:02d}"] = np.nanmean(yy) if np.isfinite(yy).any() else np.nan
        # Barnes et al. 2021 fall window (Oct 1 – Dec 1 of the fall year)
        for v in ("ndvi", "ndti", "sti"):
            yy = VARS[v][fall, j]
            r[f"fall_{v}"] = np.nanmean(yy) if np.isfinite(yy).any() else np.nan
        yy = LSTREL[lfall, j]
        r["fall_lstrel"] = np.nanmean(yy) if np.isfinite(yy).any() else np.nan
        yy = VARS["ndvi"][spr, j]
        r["spr_ndvi_max"] = np.nanmax(yy) if np.isfinite(yy).any() else np.nan
        # evidence behind the numbers
        r["n_obs_season"] = int(np.isfinite(VARS["ndvi"][sel, j]).sum())
        r["n_obs_fall"] = int(np.isfinite(VARS["ndvi"][fall, j]).sum())
        r["n_obs_lst_season"] = int(np.isfinite(LSTREL[lsel, j]).sum())
        r["n_obs_lst_fall"] = int(np.isfinite(LSTREL[lfall, j]).sum())
        rows.append(r)
    return rows


def pixel_mean(stack, sel):
    with np.errstate(all="ignore"):
        return np.nanmean(stack[sel], axis=0) if sel.any() else np.full(stack.shape[1:], np.nan)


for SITE in SITES:
    use_site(SITE)
    roi = gpd.read_file(P("roi.geojson")); utm = roi.estimate_utm_crs()
    grid30 = load_da(P("grid_30m.nc")).rio.write_crs(utm)
    labels = pd.read_csv(P("seasons_site.csv"))
    fields = gpd.read_file(P("fields_site_utm.gpkg")).sort_values("field_id").reset_index(drop=True)
    assert (fields.field_id.values == np.arange(1, len(fields) + 1)).all()
    core = read_tif(P("field_core.tif")).astype("int32")
    csb = gpd.read_file(P("csb_roi.gpkg")).reset_index(drop=True)
    csb_id = read_tif(P("csb_id.tif")).astype("int32")
    csb_px = np.bincount(csb_id.ravel(), minlength=len(csb) + 1)

    with xr.open_dataset(P("hls_bands.nc")) as _b:
        B = {b: _b[b].values.astype("float32") for b in BANDS6}
        t_opt = pd.DatetimeIndex(_b.time.values)
    with np.errstate(all="ignore"):
        PIX = dict(B)
        PIX["ndvi"] = (B["nir"] - B["red"]) / (B["nir"] + B["red"])
        PIX["gcvi"] = B["nir"] / B["green"] - 1.0
        PIX["ndti"] = (B["swir1"] - B["swir2"]) / (B["swir1"] + B["swir2"])
        PIX["sti"] = B["swir1"] / B["swir2"]
    for k in ("ndvi", "ndti"):
        PIX[k][(PIX[k] < -1) | (PIX[k] > 1)] = np.nan
    PIX["gcvi"][(PIX["gcvi"] < -1) | (PIX["gcvi"] > 20)] = np.nan
    PIX["sti"][(PIX["sti"] <= 0) | (PIX["sti"] > 5)] = np.nan

    with xr.open_dataset(P("lst_scenes.nc")) as _l:
        LST = _l.lst.values.astype("float32"); t_lst = pd.DatetimeIndex(_l.time.values)

    gt_rows, csb_rows = [], []
    for season in sorted(labels.season.unique()):
        D = season_window(season)
        cropland = read_tif(P(f"cropland_{season}.tif")).astype(bool)
        # LST relative to the scene's cropland median (cancels weather, overpass time)
        LREL = np.full_like(LST, np.nan)
        for i in range(LST.shape[0]):
            v = LST[i][cropland]
            if np.isfinite(v).mean() >= CFG["st"]["min_crop_clear"]:
                LREL[i] = LST[i] - np.nanmedian(v)
        # --- ground-truth fields (core pixels) ---
        VG = {v: unit_means(PIX[v], core, len(fields), CFG["feat"]["min_clear"]) for v in PIX}
        LG = unit_means(LREL, core, len(fields), CFG["feat"]["min_clear"])
        lab = labels[labels.season == season].set_index("field_uid")
        extra = {u: dict(cc_present=int(lab.cc_present.get(u, -1)), role=r_)
                 for u, r_ in zip(fields.field_uid, fields.role)}
        gt_rows += season_rows("gt_field", core, list(fields.field_uid), D, t_opt, VG, t_lst, LG, extra)
        # --- every CSB field big enough to be a field (for future labels) ---
        VC = {v: unit_means(PIX[v], csb_id, len(csb), CFG["feat"]["min_clear"]) for v in PIX}
        LC = unit_means(LREL, csb_id, len(csb), CFG["feat"]["min_clear"])
        ok = csb_px[1:] >= CFG["min_field_px"]
        cid = list(csb.CSBID.astype(str))
        extra = {c: dict(cropland=bool(cropland[csb_id == i + 1].mean() > 0.5)) for i, c in enumerate(cid)}
        rows = season_rows("csb_field", csb_id, cid, D, t_opt, VC, t_lst, LC, extra)
        csb_rows += [r for r, k in zip(rows, ok) if k]

        # --- per-pixel firerx layers for this season (fall window + spring max) ---
        ts = lambda s: pd.Timestamp(s.replace("y0", str(D["y0"])).replace("y1", str(D["y1"])))
        f0, f1 = ts(CFG["feat"]["fall"][0]), ts(CFG["feat"]["fall"][1])
        s0, s1 = ts(CFG["feat"]["spring_max"][0]), ts(CFG["feat"]["spring_max"][1])
        fall, spr = (t_opt >= f0) & (t_opt < f1), (t_opt >= s0) & (t_opt <= s1)
        lfall = (t_lst >= f0) & (t_lst < f1)
        with np.errstate(all="ignore"):
            px_layers = {"fall_ndvi": pixel_mean(PIX["ndvi"], fall),
                         "fall_ndti": pixel_mean(PIX["ndti"], fall),
                         "fall_sti": pixel_mean(PIX["sti"], fall),
                         "fall_lstrel": pixel_mean(LREL, lfall),
                         "spr_ndvi_max": np.nanmax(np.where(spr[:, None, None], PIX["ndvi"], np.nan), axis=0)}
        out = P(f"layers/{season}/pixel"); os.makedirs(out, exist_ok=True)
        tags = dict(season=season, fall=f"{f0.date()} → {f1.date()}", spring_max=f"{s0.date()} → {s1.date()}")
        msg = []
        for name, a in px_layers.items():
            a = a.astype("float32")
            nan_share = np.isnan(a[cropland]).mean()
            fill = np.nanmedian(a[cropland]) if np.isfinite(a[cropland]).any() else 0.0
            a = np.where(np.isfinite(a), a, fill)          # no observation → season cropland median
            a[~cropland] = 0.0                             # firerx: NaN-free, zero off cropland
            write_tif(os.path.join(out, f"{name}.tif"), a, grid30, tags)
            write_tif(P(f"layers/{season}/{name}.tif"), a, grid30, tags)
            msg.append(f"{name} (no obs {nan_share:.0%})")
        print(f"{season}: fall {f0.date()}→{f1.date()} · {int(fall.sum())} HLS days, {int(lfall.sum())} LST days"
              f" · layers {', '.join(msg)}")

    gt = pd.DataFrame(gt_rows); cs = pd.DataFrame(csb_rows)

    # --- Sentinel-1: monthly mean backscatter (dB) of the unit, Oct–May ---------
    # unit mean in linear power per scene, then dB; VH − VV is the cross ratio
    if have("s1_scenes.nc"):
        with xr.open_dataset(P("s1_scenes.nc")) as _s:
            SVV, SVH = _s.vv.values.astype("float32"), _s.vh.values.astype("float32")
            t_s1 = pd.DatetimeIndex(_s.time.values)
        s1_rows = []
        for unit, lab_, uids_ in (("gt_field", core, list(fields.field_uid)), ("csb_field", csb_id, cid)):
            with np.errstate(all="ignore"):
                dvv = 10 * np.log10(unit_means(SVV, lab_, len(uids_), CFG["feat"]["min_clear"]))
                dvh = 10 * np.log10(unit_means(SVH, lab_, len(uids_), CFG["feat"]["min_clear"]))
            for season in sorted(labels.season.unique()):
                D = season_window(season)
                ts = lambda s: pd.Timestamp(s.replace("y0", str(D["y0"])).replace("y1", str(D["y1"])))
                sel = (t_s1 >= ts(CFG["feat"]["window"][0])) & (t_s1 <= ts(CFG["feat"]["window"][1]))
                mon = t_s1[sel].month
                for j, u in enumerate(uids_):
                    r = dict(unit=unit, uid=str(u), season=season)
                    for m in MON:
                        k = mon == m
                        for nm, arr in (("vv", dvv), ("vh", dvh), ("cr", dvh - dvv)):
                            yy = arr[sel, j][k]
                            r[f"s1_{nm}_m{m:02d}"] = float(np.nanmean(yy)) if np.isfinite(yy).any() else np.nan
                    s1_rows.append(r)
        S1F = pd.DataFrame(s1_rows)
        gt = gt.assign(uid=gt.uid.astype(str)).merge(S1F[S1F.unit == "gt_field"].drop(columns="unit"),
                                                     on=["uid", "season"], how="left")
        cs = cs.assign(uid=cs.uid.astype(str)).merge(S1F[S1F.unit == "csb_field"].drop(columns="unit"),
                                                     on=["uid", "season"], how="left")

    # --- Ma et al. weather: monthly mean temperature, monthly total precipitation
    uw = pd.read_csv(P("unit_weather.csv"), dtype={"uid": str})
    wx = pd.read_csv(P("daymet_daily.csv"), parse_dates=["date"])
    wrows = []
    for season in sorted(labels.season.unique()):
        D = season_window(season)
        ts = lambda s: pd.Timestamp(s.replace("y0", str(D["y0"])).replace("y1", str(D["y1"])))
        w = wx[(wx.date >= ts(CFG["feat"]["window"][0])) & (wx.date <= ts(CFG["feat"]["window"][1]))]
        g = w.assign(m=w.date.dt.month).groupby(["key", "m"]).agg(t=("tmean", "mean"), p=("prcp", "sum"),
                                                                   n=("date", "nunique"))
        g.loc[g.n < 25, ["t", "p"]] = np.nan              # month not covered by Daymet
        for key, kg in g.groupby(level=0):
            r = dict(key=key, season=season)
            for m in MON:
                r[f"tmean_m{m:02d}"] = kg.t.get((key, m), np.nan)
                r[f"prcp_m{m:02d}"] = kg.p.get((key, m), np.nan)
            wrows.append(r)
    WX = pd.DataFrame(wrows)
    add_wx = lambda df_, unit: (df_.assign(uid=df_.uid.astype(str))
                                .merge(uw[uw.unit == unit][["uid", "key"]], on="uid", how="left")
                                .merge(WX, on=["key", "season"], how="left").drop(columns="key"))
    gt = add_wx(gt, "gt_field"); cs = add_wx(cs, "csb_field")
    gt = gt[gt.cc_present >= 0]
    gt.to_csv(P("season_features_gt.csv"), index=False)
    cs.to_csv(P("season_features_csb.csv"), index=False)
    n_ma = len([c for c in gt.columns if re.fullmatch(r"(blue|green|red|nir|swir1|swir2|ndvi|gcvi)_m\d\d_(min|max)", c)])
    n_wx = len([c for c in gt.columns if re.fullmatch(r"(tmean|prcp)_m\d\d", c)])
    n_rs = len([c for c in gt.columns if re.fullmatch(r"(ndti|sti)_m\d\d_(min|max)", c)])
    n_s1 = len([c for c in gt.columns if re.fullmatch(r"s1_(vv|vh|cr)_m\d\d", c)])
    print(f"feature groups: Ma et al. {n_ma} + {n_wx} = {n_ma + n_wx}; NDTI/STI harmonic {n_rs}; "
          f"Sentinel-1 monthly {n_s1}")
    print(f"\nfeatures per unit-season: {gt.shape[1] - 6}  (Ma et al. baseline: {n_ma} Landsat + {n_wx} weather"
          f" = {n_ma + n_wx}; weather missing in {int(gt[[c for c in gt if c.startswith('tmean_')]].isna().any(axis=1).sum())} labelled rows)")
    print(f"saved: season_features_gt.csv ({len(gt)} labelled field-seasons), "
          f"season_features_csb.csv ({len(cs)} CSB field-seasons), layers/<season>/fall_*.tif, spr_ndvi_max.tif")
    print(gt[["uid", "season", "cc_present", "n_obs_season", "n_obs_fall", "n_obs_lst_fall",
              "fall_ndti", "fall_lstrel", "spr_ndvi_max"]].round(3).to_string(index=False))


## Cell 21 — Observed trajectories and window summaries

Per season and detection window: each field's mean, maximum and number of
observations of HLS NDVI, GCVI, NDTI, STI, relative LST, Sentinel-1 VV / VH /
VH − VV and (once downloaded) PlanetScope NDVI, GCVI, NDRE; the difference of
the window means (CC − control) and the mean of the **same-day differences**
(dates when both fields were observed, the cleaner comparison). The figure
shows the observations themselves, one column per season, windows shaded
(corn window gold, harvest brown). Nothing is fitted, tested or classified.

In [ ]:
# =============================================================================
# Cell 21 — Observed trajectories and window summaries, CC field vs control
# =============================================================================
# Real observations only (no interpolation, no model): a field's value on a
# date is the mean of its clear core pixels, kept when ≥ min_clear of the core
# is clear. For every season and detection window (Cell 11b) the table gives,
# per variable, each field's mean and number of observations, the difference
# of the window means (CC − control) and the mean of the same-day differences
# (dates when both fields were observed — same scene, same illumination, same
# radar geometry). Descriptive only: no test, no classifier, no threshold.
import re, warnings
from matplotlib.lines import Line2D
warnings.filterwarnings("ignore", message="Mean of empty slice")

VAR_LABEL = {"ndvi": "NDVI (HLS)", "gcvi": "GCVI (HLS)", "ndti": "NDTI (HLS)", "sti": "STI (HLS)",
             "lstrel": "LST − cropland median (K)", "s1_vh": "S1 VH (dB)", "s1_cr": "S1 VH − VV (dB)",
             "s1_vv": "S1 VV (dB)", "ps_ndvi": "NDVI (PlanetScope)", "ps_gcvi": "GCVI (PlanetScope)",
             "ps_ndre": "NDRE (PlanetScope, 705 nm)"}
PLOT_VARS = ["ndvi", "gcvi", "ndti", "sti", "lstrel", "s1_vh", "s1_cr", "ps_ndvi", "ps_ndre"]
WIN_COLOR = {"spring": "#4a7ec4", "postharvest": "#4a7ec4", "corn": "#d9a520"}

for SITE in SITES:
    use_site(SITE)
    labels = pd.read_csv(P("seasons_site.csv"))
    fields = gpd.read_file(P("fields_site_utm.gpkg")).sort_values("field_id").reset_index(drop=True)
    core = read_tif(P("field_core.tif")).astype("int32")
    dw = pd.read_csv(P("detect_windows.csv"), parse_dates=["start", "end"])
    uids = list(fields.field_uid); n_f = len(uids); mc = CFG["feat"]["min_clear"]

    # --- field series per source ------------------------------------------------
    with xr.open_dataset(P("hls_bands.nc")) as _b:
        B = {b: _b[b].values.astype("float32") for b in ("green", "red", "nir", "swir1", "swir2")}
        t_opt = pd.DatetimeIndex(_b.time.values)
    with np.errstate(all="ignore"):
        PIXI = {"ndvi": (B["nir"] - B["red"]) / (B["nir"] + B["red"]),
                "gcvi": B["nir"] / B["green"] - 1,
                "ndti": (B["swir1"] - B["swir2"]) / (B["swir1"] + B["swir2"]),
                "sti": B["swir1"] / B["swir2"]}
    for k in ("ndvi", "ndti"):
        PIXI[k][(PIXI[k] < -1) | (PIXI[k] > 1)] = np.nan
    PIXI["gcvi"][(PIXI["gcvi"] < -1) | (PIXI["gcvi"] > 20)] = np.nan
    PIXI["sti"][(PIXI["sti"] <= 0) | (PIXI["sti"] > 5)] = np.nan
    SER = {v: (t_opt, unit_means(PIXI[v], core, n_f, mc)) for v in PIXI}     # (T, fields)
    del B, PIXI

    # LST relative to the scene's cropland median — cropland of the season the date belongs to
    with xr.open_dataset(P("lst_scenes.nc")) as _l:
        LST = _l.lst.values.astype("float32"); t_lst = pd.DatetimeIndex(_l.time.values)
    lrel = np.full((len(t_lst), n_f), np.nan, "float32")
    for season in sorted(labels.season.unique()):
        D = season_window(season)
        crop = read_tif(P(f"cropland_{season}.tif")).astype(bool)
        sel = np.where((t_lst >= pd.Timestamp(f"{D['y0']}-06-01")) & (t_lst < pd.Timestamp(f"{D['y1']}-07-01")))[0]
        for i in sel:
            v = LST[i][crop]
            if np.isfinite(v).mean() >= CFG["st"]["min_crop_clear"]:
                lrel[i] = unit_means((LST[i] - np.nanmedian(v))[None], core, n_f, mc)[0]
    SER["lstrel"] = (t_lst, lrel)

    if have("s1_scenes.nc"):
        with xr.open_dataset(P("s1_scenes.nc")) as _s:
            t_s1 = pd.DatetimeIndex(_s.time.values)
            with np.errstate(all="ignore"):
                vv = 10 * np.log10(unit_means(_s.vv.values, core, n_f, mc))
                vh = 10 * np.log10(unit_means(_s.vh.values, core, n_f, mc))
        SER.update(s1_vv=(t_s1, vv), s1_vh=(t_s1, vh), s1_cr=(t_s1, vh - vv))

    if have("ps_unit_obs.csv"):
        po = pd.read_csv(P("ps_unit_obs.csv"))
        po = po[(po.unit == "gt_field") & (po.clear >= mc)]
        if len(po):
            t_ps = pd.DatetimeIndex(sorted(po.date.unique()))
            for v in ("ndvi", "gcvi", "ndre"):
                pv = po.pivot_table(index="date", columns="uid", values=v, aggfunc="mean")
                pv.index = pd.DatetimeIndex(pv.index)
                SER[f"ps_{v}"] = (t_ps, pv.reindex(index=t_ps, columns=uids).values.astype("float32"))

    # --- window summaries and CC − control ---------------------------------------
    summ, paired = [], []
    for _, w in dw.iterrows():
        lab = labels[labels.season == w.season].set_index("field_uid")
        cc_f = [u for u in uids if lab.cc_present.get(u, -1) == 1]
        ct_f = [u for u in uids if lab.cc_present.get(u, -1) == 0]
        for v, (tt, arr) in SER.items():
            sel = (tt >= w.start) & (tt <= w.end)
            for j, u in enumerate(uids):
                y = arr[sel, j]; ok = np.isfinite(y)
                summ.append(dict(season=w.season, window=w.window, start=w.start.date(), end=w.end.date(),
                                 variable=v, field_uid=u, role=fields.role[j],
                                 cc_present=int(lab.cc_present.get(u, -1)), n_obs=int(ok.sum()),
                                 mean=float(y[ok].mean()) if ok.any() else np.nan,
                                 max=float(y[ok].max()) if ok.any() else np.nan,
                                 min=float(y[ok].min()) if ok.any() else np.nan))
            if cc_f and ct_f:
                ic = [uids.index(u) for u in cc_f]; it_ = [uids.index(u) for u in ct_f]
                a = np.nanmean(arr[sel][:, ic], axis=1) if sel.any() else np.array([])
                b = np.nanmean(arr[sel][:, it_], axis=1) if sel.any() else np.array([])
                both = np.isfinite(a) & np.isfinite(b)
                with np.errstate(all="ignore"):
                    paired.append(dict(season=w.season, window=w.window, variable=v,
                                       cc_mean=np.nanmean(a) if np.isfinite(a).any() else np.nan,
                                       ctrl_mean=np.nanmean(b) if np.isfinite(b).any() else np.nan,
                                       n_cc=int(np.isfinite(a).sum()), n_ctrl=int(np.isfinite(b).sum()),
                                       n_same_day=int(both.sum()),
                                       same_day_diff=float(np.mean(a[both] - b[both])) if both.any() else np.nan,
                                       same_day_cc_higher=f"{int((a[both] > b[both]).sum())}/{int(both.sum())}"))
    summ = pd.DataFrame(summ); paired = pd.DataFrame(paired)
    paired["diff_of_means"] = paired.cc_mean - paired.ctrl_mean
    summ.to_csv(P("window_summary.csv"), index=False)
    paired.to_csv(P("window_paired.csv"), index=False)

    print(f"{SITE}: CC field − control per detection window (observed dates only; descriptive)\n")
    for (s, wn), g in paired.groupby(["season", "window"], sort=False):
        ww = dw[(dw.season == s) & (dw.window == wn)].iloc[0]
        print(f"{s}  {wn}  {ww.start:%Y-%m-%d} → {ww.end:%Y-%m-%d}")
        with pd.option_context("display.float_format", "{:+.3f}".format, "display.width", 200):
            print(g[["variable", "cc_mean", "ctrl_mean", "diff_of_means", "n_cc", "n_ctrl", "n_same_day",
                     "same_day_diff", "same_day_cc_higher"]].to_string(index=False))
        print()

    # --- figure: observed points, one column per season ---------------------------
    seasons = sorted(labels.season.unique())
    rows_v = [v for v in PLOT_VARS if v in SER and np.isfinite(SER[v][1]).any()]
    fig, ax = plt.subplots(len(rows_v), len(seasons), figsize=(5.6 * len(seasons), 2.2 * len(rows_v)),
                           sharey="row", squeeze=False)
    for j, s in enumerate(seasons):
        D = season_window(s); y0, y1 = D["y0"], D["y1"]
        ws = dw[dw.season == s]
        x0 = pd.Timestamp(f"{y0}-06-01") if (ws.window == "corn").any() else pd.Timestamp(f"{y0}-09-15")
        x1 = pd.Timestamp(f"{y1}-06-15")
        lab = labels[labels.season == s].set_index("field_uid")
        seed = pd.to_datetime(lab.cc_seed_date.dropna().iloc[0]) if lab.cc_seed_date.notna().any() else None
        for r_, v in enumerate(rows_v):
            a = ax[r_, j]; tt, arr = SER[v]
            sel = (tt >= x0) & (tt <= x1)
            for k, u in enumerate(uids):
                y = arr[sel, k]; ok = np.isfinite(y)
                cc = lab.cc_present.get(u, -1) == 1
                a.plot(tt[sel][ok], y[ok], "o-", ms=3, lw=0.8, color="#2b7a3d" if cc else "#8a99a5",
                       label=f"{u.split('_')[-1]} ({'CC' if cc else 'no CC'})" if (r_ == 0) else None)
            for _, w in ws.iterrows():
                a.axvspan(w.start, w.end, color=WIN_COLOR.get(w.window, "#4a7ec4"), alpha=0.12)
            if seed is not None:
                a.axvline(seed, color="#2b7a3d", ls="--", lw=0.9)
            a.axvline(D["term"], color="#c44", ls="--", lw=0.9)
            if (ws.window == "corn").any():
                cw = ws[ws.window == "corn"].iloc[0]; ph = ws[ws.window == "postharvest"].iloc[0]
                a.axvspan(cw.end, ph.start, color="#8b5a2b", alpha=0.10)
            if v in ("lstrel",):
                a.axhline(0, color="k", lw=0.5)
            a.set_xlim(x0, x1); a.grid(alpha=0.3)
            a.xaxis.set_major_locator(mdates.MonthLocator()); a.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
            a.tick_params(axis="x", labelsize=7, rotation=90)
            if j == 0:
                a.set_ylabel(VAR_LABEL.get(v, v), fontsize=8)
        ttl = " · ".join(f"{w.window} {w.start:%m-%d}→{w.end:%m-%d}" for _, w in ws.iterrows())
        ax[0, j].set_title(f"{s}\n{ttl}", fontsize=9)
    ax[0, 0].legend(fontsize=7, loc="upper right")
    handles = [Line2D([0], [0], color="#4a7ec4", lw=6, alpha=0.3, label="detection window (spring / post-harvest)"),
               Line2D([0], [0], color="#d9a520", lw=6, alpha=0.3, label="corn window (interseeded)"),
               Line2D([0], [0], color="#8b5a2b", lw=6, alpha=0.3, label="harvest (between windows)"),
               Line2D([0], [0], color="#2b7a3d", ls="--", label="cover-crop seeding"),
               Line2D([0], [0], color="#c44", ls="--", label="termination")]
    fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=8, frameon=False)
    fig.suptitle(f"{SITE}: observed values only (no gap filling)", fontsize=11)
    plt.tight_layout(rect=(0, 0.03, 1, 0.98))
    fig_path = os.path.join(FIG_DIR, f"trajectories_obs_{SITE}.png")
    plt.savefig(fig_path, dpi=150, bbox_inches="tight"); plt.show()
    print(f"saved: window_summary.csv, window_paired.csv, {fig_path}")


## Cell 22 — Archive to the Data Store

Products, `layers/`, figures and both per-scene caches go to
`ground_truth_runs/<site>/`. Unchanged files are skipped.

v9 adds `hls_bands.nc`, the LST stack, the feature tables and the `hls_v3` and `lst` caches.

v10 adds the Daymet tables and cache, and rewrites a layer or product only when it is newer than its archived copy.

v11 adds the detection windows, the Sentinel-1 and PlanetScope tables and caches, and the window summaries.

In [ ]:
# =============================================================================
# Cell 22 — Copy each site's products to the Data Store
# =============================================================================
def _rm(path):
    """Delete on the iRODS mount. It sometimes reports "Remote I/O error" for a
    delete that did happen, or lists a file that is already gone, so the
    outcome is checked rather than the call."""
    for _ in range(3):
        try:
            os.remove(path)
        except FileNotFoundError:
            return
        except OSError:
            time.sleep(2)
        if not os.path.exists(path):
            return
    raise OSError(f"could not delete {path}")


def put(src, rel, force=False):
    """Copy one file into the archive unless an identical-size copy is there
    (`force` rewrites anyway: small products whose size may not change).

    The Data Store mount refuses to overwrite a file in place, so a changed
    file is deleted first; the copy is retried once and verified by size."""
    dst = os.path.join(ARCHIVE, rel)
    mb = os.path.getsize(src) / 1e6
    if os.path.exists(dst):
        if force == "newer":        # rewrite only what changed since the last archive run
            force = os.path.getmtime(src) > os.path.getmtime(dst) + 1
        if not force and os.path.getsize(dst) == os.path.getsize(src):
            return False, mb
        _rm(dst)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    want = os.path.getsize(src)
    for attempt in range(3):
        try:
            shutil.copyfile(src, dst)
        except OSError:
            time.sleep(3)
        # the mount can report a stale size for a few seconds after a write
        for _ in range(6):
            if os.path.exists(dst) and os.path.getsize(dst) == want:
                return True, mb
            time.sleep(2)
        if os.path.exists(dst):
            _rm(dst)
    raise OSError(f"could not archive {rel}")


for SITE in SITES:
    use_site(SITE)
    os.makedirs(ARCHIVE, exist_ok=True)
    PRODUCTS = ["roi.geojson", "windows.csv", "cfg_site.json", "fields_site.gpkg",
                "fields_site_utm.gpkg", "seasons_site.csv", "grid_30m.nc", "grid_70m.nc",
                "hls_index.csv", "eco_index.csv", "hls_scenes.nc", "fused_cube.nc",
                "season_windows.csv", "field_id.tif", "field_core.tif", "csb_roi.gpkg", "csb_id.tif",
                "fields_features.gpkg", "layer_values_by_season.csv", "layer_paired_diff.csv",
                "hls_bands.nc", "lst_index.csv", "lst_scenes.nc", "season_features_gt.csv",
                "season_features_csb.csv", "detect_windows.csv", "s1_index.csv", "s1_scenes.nc",
                "ps_index.csv", "ps_calibration.csv", "ps_unit_obs.csv",
                "window_summary.csv", "window_paired.csv",
                "unit_weather.csv", "daymet_daily.csv"]
    PRODUCTS += sorted(os.path.basename(f) for f in glob.glob(P("cropland_*.tif")))
    t0, copied, skipped, missing = time.time(), [], 0, []
    for name in PRODUCTS:
        if not os.path.exists(P(name)):
            missing.append(name); continue
        did, mb = put(P(name), name, force="newer" if name.endswith((".csv", ".json", ".gpkg", ".tif")) else False)
        copied.append((name, mb)) if did else None; skipped += (not did)
    # layers/ and figures/ are small and change on every run: always compared
    n = n_new = 0
    for root, _, files in os.walk(P("layers")):
        for f in files:
            src = os.path.join(root, f); n += 1
            n_new += put(src, os.path.relpath(src, WORK), force="newer")[0]
    copied.append((f"layers/  ({n_new} written of {n})", 0.0))
    for sub in ("figures", "cache/hls_v2", "cache/hls_v3", "cache/eco", "cache/lst", "cache/daymet", "cache/s1", "cache/ps"):
        d = P(sub); n = n_new = 0
        if os.path.isdir(d):
            # one listing of the archive instead of a stat per file (a stat on
            # the Data Store mount costs about a second); cache files never change
            dst_dir = os.path.join(ARCHIVE, sub)
            already = set(os.listdir(dst_dir)) if os.path.isdir(dst_dir) else set()
            for f in sorted(os.listdir(d)):
                n += 1
                if f in already and not sub.startswith("figures"):
                    continue
                n_new += put(os.path.join(d, f), os.path.join(sub, f),
                             force=sub == "figures")[0]
        if n:
            copied.append((f"{sub}/  ({n_new} new of {n})", 0.0))
    print(f"archived to {ARCHIVE}   ({(time.time()-t0)/60:.1f} min)")
    for name, mb in copied:
        print(f"  {name:<44} {mb:>8.2f} MB" if mb else f"  {name}")
    print(f"  {'already archived, unchanged':<44} {skipped:>5} file(s)")
    if missing:
        print(f"not found (run the earlier cells): {missing}")


## Cell 23 — All sites on one page

One row per site: spring NDVI with the labelled fields, field trajectories,
and the AUC of every layer. Tables of layer values and AUCs across sites go
to `_summary/`.

In [ ]:
# =============================================================================
# Cell 23 — Combined figure and tables across all sites
# =============================================================================
SUMMARY = os.path.join(CFG["work_root"], "_summary")
os.makedirs(SUMMARY, exist_ok=True)
ROLE_COLOR = {"cc_treatment": "#2b7a3d", "candidate_same_farm": "#7fbf7b",
              "nt_control": "#8a99a5", "neighbour_control": "#8a99a5",
              "trial_host_field": "#c4763a"}

done = [s for s in SITES if os.path.isfile(os.path.join(CFG["work_root"], s, "layer_paired_diff.csv"))]
assert done, "no site has reached Cell 17 yet"
feat_all, diff_all = [], []
fig, axes = plt.subplots(len(done), 3, figsize=(19, 4.8 * len(done)),
                         gridspec_kw={"width_ratios": [1, 2.2, 1.1]}, squeeze=False)

for k, site in enumerate(done):
    use_site(site, quiet=True)
    with xr.open_dataset(P("fused_cube.nc")) as _f:
        fused = _f.load()
    fields = gpd.read_file(P("fields_site_utm.gpkg"))
    labels = pd.read_csv(P("seasons_site.csv"),
                         parse_dates=["detect_start", "detect_end", "best_start",
                                      "best_end", "cc_termination_date"])
    windows = pd.read_csv(P("windows.csv"))
    fid  = rxr.open_rasterio(P("field_id.tif")).squeeze(drop=True).values
    core = rxr.open_rasterio(P("field_core.tif")).squeeze(drop=True).values
    dtab = pd.read_csv(P("layer_paired_diff.csv"), index_col="layer")
    ft = pd.read_csv(P("layer_values_by_season.csv")); ft.insert(0, "site", site)
    feat_all.append(ft); diff_all.append(dtab.assign(site=site))
    t = pd.DatetimeIndex(fused.time.values)
    ndvi = fused.ndvi.values
    cen = gpd.read_file(P("roi.geojson")).geometry.centroid.iloc[0]

    # --- header text for this row ------------------------------------------
    n_cc, n_no = int((labels.cc_present == 1).sum()), int((labels.cc_present == 0).sum())
    wd = " + ".join(f"{d} d" for d in sorted(windows.win_days.unique(), reverse=True) if d in (8, 16))
    hdr = (f"{site}   {abs(cen.y):.3f}°{'N' if cen.y >= 0 else 'S'} {abs(cen.x):.3f}°{'W' if cen.x < 0 else 'E'}   "
           f"{len(fields)} fields · {CFG['date_start']} → {CFG['date_end']} · windows {wd}"
           f" · {n_cc} CC / {n_no} no-CC field-seasons · ET {'yes' if 'et' in fused else 'no'}")

    # --- panel 1: NDVI map, strongest labelled spring window -----------------
    pos = labels[labels.cc_present == 1]
    last = (pos if len(pos) else labels).sort_values("detect_end").iloc[-1]
    spring_t = last.best_end if pd.notna(last.best_end) else last.detect_end
    i_sp = int(np.argmin(np.abs(t - spring_t)))
    edge = np.zeros_like(fid, bool)
    edge[:, 1:] |= fid[:, 1:] != fid[:, :-1]; edge[1:, :] |= fid[1:, :] != fid[:-1, :]
    a = axes[k, 0]
    im = a.imshow(ndvi[i_sp], cmap="RdYlGn", vmin=0, vmax=0.8)
    a.imshow(np.ma.masked_where(~edge, np.ones_like(fid, "float32")), cmap="gray_r",
             vmin=0, vmax=1, alpha=0.6, interpolation="nearest")
    for _, f in fields.iterrows():
        yy, xx = np.where(fid == f.field_id)
        if len(yy):
            a.text(xx.mean(), yy.mean(), f.field_uid.split("_")[-1], fontsize=7, ha="center",
                   va="center", color=ROLE_COLOR.get(f.role, "k"),
                   bbox=dict(fc="white", ec="none", alpha=0.6, pad=0.5))
    a.set_title(f"NDVI {t[i_sp]:%Y-%m-%d} (spring, season {last.season})", fontsize=9)
    a.set_xticks([]); a.set_yticks([])
    plt.colorbar(im, ax=a, fraction=0.046)

    # --- panel 2: field trajectories ----------------------------------------
    a = axes[k, 1]
    for _, f in fields.iterrows():
        m = core == f.field_id
        a.plot(t, np.nanmean(ndvi[:, m], axis=1), "-", color=ROLE_COLOR.get(f.role, "k"),
               lw=1.3 if f.role in ("cc_treatment", "candidate_same_farm") else 0.9,
               alpha=0.9 if f.role == "cc_treatment" else 0.6,
               label=f"{f.field_uid.split('_')[-1]} ({f.role})")
    for s in sorted(labels.season.unique()):          # the detection windows
        Dw = season_window(s)
        a.axvspan(Dw["start"], Dw["end"], color="#4a7ec4", alpha=0.12)
        a.axvline(Dw["term"], color="#c44", lw=0.8, ls="--")
    a.xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 4, 7, 10)))
    a.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    a.xaxis.set_minor_locator(mdates.MonthLocator())
    a.tick_params(axis="x", labelsize=7, rotation=60)
    a.set_ylabel("field-mean NDVI"); a.grid(alpha=0.35, which="major"); a.grid(alpha=0.12, which="minor")
    a.set_title(hdr, fontsize=9.5, loc="left")
    if len(fields) <= 12:
        a.legend(fontsize=6.5, ncol=3, loc="upper left")

    # --- panel 3: CC field − control per layer and season (descriptive) ------
    a = axes[k, 2]
    vals_ = dtab.values.astype(float)
    a.imshow(np.sign(vals_), cmap="PiYG", vmin=-1.6, vmax=1.6, aspect="auto")
    for (ii, jj), val in np.ndenumerate(vals_):
        a.text(jj, ii, f"{val:+.3g}", ha="center", va="center", fontsize=7)
    a.set_xticks(range(dtab.shape[1])); a.set_xticklabels(dtab.columns, fontsize=8)
    a.set_yticks(range(dtab.shape[0])); a.set_yticklabels(dtab.index, fontsize=8)
    a.set_title("v8 layers: CC field − control, per season", fontsize=9)

plt.tight_layout()
fig_path = os.path.join(SUMMARY, "all_sites_summary.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight"); plt.show()

pd.concat(feat_all).to_csv(os.path.join(SUMMARY, "layer_values_all_sites.csv"), index=False)
diff_long = pd.concat(diff_all)
diff_long.to_csv(os.path.join(SUMMARY, "layer_paired_diff_all_sites.csv"))
print(f"{len(done)} site(s) summarised → {fig_path}")
with pd.option_context("display.float_format", "{:+.3f}".format):
    print("\nv8 layers, CC − control per season:\n" + diff_long.to_string())

if CFG["use_archive"]:
    dst = os.path.join(CFG["archive_root"], "_summary")
    os.makedirs(dst, exist_ok=True)
    for f in os.listdir(SUMMARY):
        shutil.copy2(os.path.join(SUMMARY, f), os.path.join(dst, f))
    print(f"copied to {dst}")

---
## Reading the results

**Descriptive, on purpose.** Two fields over three seasons show what the records
look like; they cannot establish that a feature detects cover crops. The same
code runs unchanged once more labelled fields are joined to
`season_features_csb.csv`.

**Same-day differences first.** The two fields are 300 m apart: in a given scene
they share weather, illumination and radar geometry, so the same-day difference
is cleaner than the difference of window means (which also mixes dates).

**A difference is not yet a cover-crop effect.** v9–v10 showed a year-round
offset between the two fields (e.g. October SWIR before any cover crop was
up). Compare each window with the months before seeding in the trajectories.

**The corn window is the hard one.** Under a closed corn canopy the interseeded
cover crop is invisible; GCVI during senescence (September to harvest) is where
a difference could first appear.
